# Beyond FIFA potential: finding hidden gems before the breakthrough

I compared a player model with FIFA potential to find future 80+ players while they were still rated below 70.

In [ ]:
ROOM_TEMPLATE='<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Beyond FIFA Potential · Scouting Replay</title><style>\n*{box-sizing:border-box}body{margin:0;background:#f2f5f6;color:#173441;font:14px/1.6 system-ui,-apple-system,Arial,sans-serif}.room{padding:18px}h2{font-size:24px;margin:0 0 6px}p{margin:7px 0}.muted{font-size:12px;color:#627b86}.controls{display:flex;gap:11px;flex-wrap:wrap;align-items:end;padding:15px;background:white;border:1px solid #dce5e9;border-radius:9px;margin:15px 0}label{display:grid;gap:4px;font-size:11px;font-weight:700;color:#58717e}select,input,button{font:inherit;color:#173441;padding:8px;border:1px solid #c5d2d9;border-radius:5px;background:white}select{max-width:200px}button{cursor:pointer}input[type=checkbox]{width:16px;height:16px}.toggle{display:flex;gap:6px;align-items:center;font-size:12px}.cards{display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin:14px 0}.stat{background:white;border:1px solid #dce5e9;border-top:3px solid #008777;border-radius:7px;padding:12px}.stat b{display:block;font-size:23px}.stat small{color:#627b86;font-size:11px}.card{background:white;border:1px solid #dce5e9;border-radius:9px;padding:15px;margin:14px 0}.grid{display:grid;grid-template-columns:1fr 1fr;gap:14px}.plot{height:450px}.smallplot{height:300px}h3{font-size:17px;margin:0 0 7px}.tablewrap{max-height:320px;overflow:auto}table{width:100%;border-collapse:collapse;font-size:12px}th,td{padding:8px;border-bottom:1px solid #e1e8ec;text-align:left;white-space:nowrap}th{position:sticky;top:0;background:#f0f5f6}tbody tr{cursor:pointer}tbody tr:hover{background:#edf7f3}.green{color:#008777;font-weight:700}.red{color:#bc625e}.notice{padding:10px 14px;background:#fff5e4;border-left:3px solid #c48b29;font-size:12px;margin:13px 0}.playerhead{font-size:20px;font-weight:700}.hidden{display:none}.search{width:240px}.tools{display:flex;gap:12px;align-items:center;justify-content:space-between;flex-wrap:wrap}.foot{font-size:11px;color:#627b86;margin-top:15px}#loading{padding:45px;text-align:center;color:#58717e}@media(max-width:800px){.grid{grid-template-columns:1fr}.cards{grid-template-columns:1fr 1fr}.room{padding:10px}}\n</style><script>__PLOTLY__</script></head><body><div id="loading">Opening the dated scouting replay…</div><main id="room" class="room hidden">\n<h2>Explore the scouting lists</h2><p class="muted">Compare the player model with FIFA potential, then inspect the recorded career. Filters rerank the full eligible pool.</p>\n<div class="controls"><label>Edition<select id="year"></select></label><label>Position<select id="role"><option value="all">All positions</option></select></label><label>Nationality<select id="country"><option value="all">All nationalities</option></select></label><label>FIFA potential ceiling<select id="ceiling"><option value="100">All potentials</option><option value="82">82 or below</option><option value="79">Below 80</option><option value="75">75 or below</option></select></label><label>Review budget<select id="budget"><option>25</option><option selected>50</option><option>100</option><option>200</option><option>500</option></select></label><label>Ranking<select id="model"></select></label><label>Map colour<select id="colour"><option value="outcome">Future outcome</option><option value="cluster">2015 profile groups</option></select></label><label class="toggle"><input type="checkbox" id="reveal" checked>Reveal future</label></div>\n<label class="toggle"><input type="checkbox" id="never70">Only players never previously rated 70+</label><div id="notice" class="notice"></div><div id="cards" class="cards"></div>\n<div class="card"><h3>Player similarity map</h3><div id="map" class="plot"></div><p id="mapnote" class="muted"></p></div>\n<div class="card"><div id="playerhead" class="playerhead"></div><p id="playernote" class="muted"></p><div class="grid"><div><h3>The career, revealed after prediction</h3><div id="career" class="smallplot"></div></div><div><h3>Why the player model prioritised this profile</h3><div id="shap" class="smallplot"></div><p id="shapnote" class="muted"></p></div></div><p id="neighbors" class="muted"></p></div>\n<div class="card"><div class="tools"><h3>The shortlist</h3><input id="search" class="search" placeholder="Search this edition\'s players"><button id="download">Show rankings CSV</button></div><p class="muted">The table displays up to 50 rows; metrics use the entire review budget. Search changes the table only. Click a name or point for its career.</p><div class="tablewrap"><table><thead><tr><th>Player</th><th>OVR</th><th>POT</th><th>ML rank</th><th>FIFA rank</th><th>Peak in window</th><th>Outcome</th><th>Club</th></tr></thead><tbody id="players"></tbody></table></div><div id="csvpanel" class="hidden"><p id="csvnote" class="muted"></p><textarea id="csvtext" aria-label="Filtered rankings CSV" readonly style="width:100%;height:160px;font:11px/1.5 monospace;padding:10px;border:1px solid #c5d2d9;border-radius:5px"></textarea></div></div>\n<p class="foot">Current rating below 70; any age; no prior observed 85+. Coordinate system and six profile groups fitted on 2015 attributes. Future labels only colour the reveal. Scores are ranking indices, not calibrated probabilities. The FC26 list uses the September 2025 source snapshot. The FC26 future outcomes are unconfirmed.</p></main><script id="packed" type="application/octet-stream">__PACKED__</script><script>__LOGIC__</script></body></html>\n'
ROOM_LOGIC='let DATA,all,byid,current=[],currentRanking=[],currentFifa=[],selected=null,selectedMethod;\nconst $=id=>document.getElementById(id),INK=\'#173441\',TEAL=\'#008777\',GOLD=\'#c48b29\',RED=\'#bc625e\';\nconst esc=s=>String(s??\'\').replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\nconst layout={font:{family:\'Arial\',color:INK,size:11},paper_bgcolor:\'white\',plot_bgcolor:\'white\',margin:{l:55,r:25,t:25,b:55},hoverlabel:{font:{size:12}},showlegend:true};\nconst config={responsive:true,displaylogo:false,modeBarButtonsToRemove:[\'lasso2d\',\'select2d\']};\nconst options=(id,items,selectedValue)=>{$(id).innerHTML=items.map(([v,t])=>`<option value="${esc(v)}">${esc(t)}</option>`).join(\'\');if(selectedValue!==undefined)$(id).value=selectedValue;};\nfunction metrics(r,k){let top=r.slice(0,k),hits=top.filter(p=>p.event_quality80===1).length,unknown=top.filter(p=>p.event_quality80===null).length,total=r.filter(p=>p.event_quality80===1).length;return{hits,unknown,total,n:top.length,precision:top.length?hits/top.length:0,recall:total?hits/total:null};}\nfunction ranking(pool,m){return [...pool].sort((a,b)=>a[\'rank\'+m]-b[\'rank\'+m]);}\nfunction lineCard(label,value,note){return `<div class="stat"><small>${esc(label)}</small><b>${esc(value)}</b><small>${esc(note)}</small></div>`;}\nfunction update(){\n let year=+$(\'year\').value,role=$(\'role\').value,country=$(\'country\').value,cap=+$(\'ceiling\').value,k=+$(\'budget\').value;\n selectedMethod=+$(\'model\').value;current=all.filter(p=>p.edition===year&&(role===\'all\'||p.position_group===role)&&(country===\'all\'||p.nationality_name===country)&&p.potential<=cap&&(!$(\'never70\').checked||p.peak_seen<70));$(\'reveal\').disabled=year>=2024;\n currentRanking=ranking(current,selectedMethod);currentFifa=ranking(current,DATA.fifa);let m=metrics(currentRanking,k),f=metrics(currentFifa,k),mature=year<2024,reveal=$(\'reveal\').checked&&mature;let rankingLabel=selectedMethod===DATA.base?\'ML\':\'Selected\';\n $(\'cards\').innerHTML=lineCard(\'Eligible players\',current.length.toLocaleString(),\'Full filtered pool\')+lineCard(\'Confirmed \'+rankingLabel+\' / FIFA\',reveal?m.hits+\' / \'+f.hits:\'—\',reveal?`Among ${m.n} reviews each`:\'Reveal the future to inspect\')+lineCard(rankingLabel+\' precision / confirmed recall\',reveal?(100*m.precision).toFixed(1)+\'% / \'+(m.recall===null?\'—\':(100*m.recall).toFixed(1)+\'%\'):\'—\',reveal?m.total+\' documented 80+ cases\':\'No mature outcome claim\')+lineCard(\'Unresolved \'+rankingLabel+\' / FIFA\',reveal?m.unknown+\' / \'+f.unknown:\'—\',\'Unknowns keep their shortlist slots\');\n $(\'notice\').textContent=mature?\'Predictions use information available in this edition. Revealed future outcomes are shown for evaluation, never used to draw the coordinates.\':\'FC26 snapshot watchlist: the next three editions are not available. These names are unconfirmed prospects, not validated discoveries.\';\n $(\'csvpanel\').classList.add(\'hidden\');drawMap(reveal,k);drawTable();selected=current.find(p=>p.player_id===selected?.player_id)||currentRanking[0]||null;drawPlayer(reveal);\n}\nfunction drawMap(reveal,k){\n let pool=current.filter(p=>p.position_group!==\'GK\');if(!pool.length){Plotly.react(\'map\',[],{...layout,annotations:[{text:\'The attribute map covers outfield players. Shortlist counts still include the selected position.\',showarrow:false,x:.5,y:.5,xref:\'paper\',yref:\'paper\'}]},config);return;}\n let chosen=currentRanking.slice(0,k).filter(p=>p.position_group!==\'GK\'),chosenIDs=new Set(chosen.map(p=>p.player_id)),fifa=currentFifa.slice(0,k).filter(p=>p.position_group!==\'GK\'&&!chosenIDs.has(p.player_id));\n let step=Math.max(1,Math.ceil(pool.length/1000)),background=pool.filter((p,i)=>i%step===0),traces=[];\n const hover=\'<b>%{customdata[1]}</b><br>OVR %{customdata[2]} · POT %{customdata[3]}<br>%{customdata[4]}<extra></extra>\';\n const trace=(p,name,marker,labels=false)=>({type:\'scatter\',mode:labels?\'markers+text\':\'markers\',x:p.map(a=>a.x),y:p.map(a=>a.y),customdata:p.map(a=>[a.player_id,a.short_name,a.overall,a.potential,a.club_name]),name,marker,hovertemplate:hover,text:labels?p.map((a,i)=>i<8?a.short_name:\'\'):undefined,textposition:\'top center\',textfont:{size:10,color:INK},cliponaxis:false});\n let colour=$(\'colour\').value,colourValue=background.map(p=>colour===\'cluster\'?p.cluster:colour===\'fit\'?p.learned_future_advantage:0);\n traces.push(trace(background,\'Other eligible profiles\',{size:5,opacity:.32,color:colour===\'outcome\'?\'#acbac2\':colourValue,colorscale:colour===\'fit\'?\'BrBG\':\'Viridis\',showscale:colour!==\'outcome\',colorbar:{title:colour===\'fit\'?\'Fit advantage\':\'Profile group\',thickness:10}}));\n if(reveal){let positive=pool.filter(p=>p.event_quality80===1&&!chosenIDs.has(p.player_id));traces.push(trace(positive,\'80+ outside ML list\',{size:8,color:TEAL,symbol:\'diamond\',opacity:.65}));}\n traces.push(trace(fifa,\'FIFA-only selections\',{size:10,color:GOLD,symbol:\'circle-open\',line:{width:1.5}},false));\n traces.push(trace(chosen,selectedMethod===DATA.base?\'ML selections\':\'Selected ranking\',{size:9,opacity:.95,color:chosen.map(p=>reveal?(p.event_quality80===1?TEAL:p.event_quality80===0?RED:\'#8b9da7\'):INK),line:{color:INK,width:.5}},true));\n let ready=Plotly.react(\'map\',traces,{...layout,height:450,xaxis:{title:{text:\'PC1 · \'+(100*DATA.variance[0]).toFixed(1)+\'% of profile variation\'},zeroline:false},yaxis:{title:{text:\'PC2 · \'+(100*DATA.variance[1]).toFixed(1)+\'%\'},zeroline:false},legend:{orientation:\'h\',y:1.12,x:0},margin:{...layout.margin,t:45}},config);\n Promise.resolve(ready).then(()=>{$(\'map\').removeAllListeners?.(\'plotly_click\');$(\'map\').on(\'plotly_click\',e=>{let id=e.points[0].customdata?.[0];let p=current.find(a=>a.player_id===id);if(p){selected=p;drawPlayer($(\'reveal\').checked&&+$(\'year\').value<2024);}});});\n $(\'mapnote\').textContent=\'The two axes retain \'+(100*DATA.variance.reduce((a,b)=>a+b,0)).toFixed(1)+\'% of variation. Names label the first eight ML selections. Green/red/grey reveal confirmed 80+, known non-target and unresolved selections. Goalkeepers are omitted from this outfield projection, not from the ranking.\';\n}\nfunction drawTable(){let reveal=$(\'reveal\').checked&&+$(\'year\').value<2024,term=$(\'search\').value.trim().toLowerCase(),mr=new Map(currentRanking.map((p,i)=>[p.player_id,i+1])),fr=new Map(currentFifa.map((p,i)=>[p.player_id,i+1]));\n let shown=currentRanking.filter(p=>!term||p.short_name.toLowerCase().includes(term)).slice(0,50);\n $(\'players\').innerHTML=shown.map(p=>`<tr data-id="${p.player_id}"><td>${esc(p.short_name)}</td><td>${p.overall}</td><td>${p.potential}</td><td>${mr.get(p.player_id)}</td><td>${fr.get(p.player_id)}</td><td>${reveal&&p.observed_peak3!==null?p.observed_peak3:\'—\'}</td><td class="${reveal&&p.event_quality80===1?\'green\':\'\'}">${reveal?(p.event_quality80===1?\'Confirmed 80+\':p.event_quality80===0?\'Known non-target\':\'Unresolved\'):\'Not revealed\'}</td><td>${esc(p.club_name)}</td></tr>`).join(\'\');\n $(\'players\').querySelectorAll(\'tr\').forEach(tr=>tr.onclick=()=>{selected=current.find(p=>p.player_id===+tr.dataset.id);drawPlayer(reveal);});\n}\nfunction drawPlayer(reveal){if(!selected){$(\'playerhead\').textContent=\'No players match these filters.\';Plotly.react(\'career\',[],layout,config);Plotly.react(\'shap\',[],layout,config);return;}\n let p=selected,year=+$(\'year\').value,mr=currentRanking.findIndex(a=>a.player_id===p.player_id)+1,fr=currentFifa.findIndex(a=>a.player_id===p.player_id)+1;\n $(\'playerhead\').textContent=p.short_name+\' · \'+p.position_group+\' · OVR \'+p.overall+\' · POT \'+p.potential;\n $(\'playernote\').textContent=p.club_name+\' · age \'+p.age+\' · \'+(selectedMethod===DATA.base?\'ML\':\'Selected\')+\' rank \'+mr+\' · FIFA rank \'+fr+\'. \'+(reveal?(p.event_quality80===1?\'Confirmed 80+ within three editions.\':p.event_quality80===0?\'Complete window; did not reach 80+.\':\'Future outcome unresolved.\'):\'Future outcome hidden or unavailable.\');\n let history=(DATA.histories[String(p.player_id)]||[]).filter(a=>a[0]<=(reveal?year+3:year)),career=[{type:\'scatter\',mode:\'lines+markers\',x:history.map(a=>a[0]),y:history.map(a=>a[1]),line:{color:INK,width:2.5},marker:{size:6},name:\'Overall\'}];\n Plotly.react(\'career\',career,{...layout,height:300,showlegend:false,xaxis:{title:{text:\'Game edition · target window ends \'+(year+3)},dtick:1},yaxis:{title:{text:\'Overall\'},range:[Math.min(45,...history.map(a=>a[1]))-2,92]},shapes:[{type:\'rect\',x0:year+.02,x1:year+3,y0:0,y1:1,yref:\'paper\',fillcolor:\'#e3eaed\',opacity:.35,line:{width:0},layer:\'below\'},{type:\'line\',x0:year,x1:year,y0:0,y1:1,yref:\'paper\',line:{color:TEAL,dash:\'dash\'}},{type:\'line\',x0:year+3,x1:year+3,y0:0,y1:1,yref:\'paper\',line:{color:\'#a7b6bd\',dash:\'dot\'}},{type:\'line\',x0:year-2,x1:year+3,y0:80,y1:80,line:{color:GOLD,dash:\'dot\'}}]},config);\n let explainMethod=[DATA.base,DATA.future].includes(selectedMethod)?selectedMethod:DATA.base,ex=DATA.explanations[year+\':\'+p.player_id+\':\'+explainMethod];\n if(ex){let feats=[...ex.features,[\'Other contributions\',ex.other]].sort((a,b)=>a[1]-b[1]);Plotly.react(\'shap\',[{type:\'bar\',orientation:\'h\',y:feats.map(a=>a[0]),x:feats.map(a=>a[1]),marker:{color:feats.map(a=>a[1]>=0?TEAL:RED)}}],{...layout,height:300,showlegend:false,margin:{l:190,r:20,t:15,b:40},xaxis:{title:{text:\'Raw score contribution\'}},yaxis:{automargin:true}},config);$(\'shapnote\').textContent=DATA.methods[explainMethod]+\': expected score \'+ex.expected.toFixed(2)+\' + all contributions = \'+ex.score.toFixed(2)+\'. This explains a ranking score, not a calibrated star probability.\';}\n else{Plotly.react(\'shap\',[],{...layout,height:300,annotations:[{text:\'Detailed SHAP is included for global top-120 lists and confirmed 80+ cases.\',showarrow:false,x:.5,y:.5,xref:\'paper\',yref:\'paper\',font:{size:12}}]},config);$(\'shapnote\').textContent=\'The full source supports explaining another player. Missing explanation detail is not a model-confidence signal.\';}\n let nn=year===2023?DATA.neighbors[String(p.player_id)]:null;$(\'neighbors\').textContent=nn?\'Closest 75+ rated peers across all \'+nn[0].dimensions+\' normalized skill dimensions: \'+nn.map(n=>n.neighbor+\' (\'+n.neighbor_overall+\', distance \'+n.distance.toFixed(2)+\')\').join(\' · \')+\'. Similarity describes style, not destiny.\':\'\';\n}\nfunction exportCSV(){let fr=new Map(currentFifa.map((p,i)=>[p.player_id,i+1])),reveal=$(\'reveal\').checked&&+$(\'year\').value<2024;let rows=[[\'player_id\',\'edition\',\'player\',\'overall\',\'potential\',\'position\',\'ml_rank\',\'fifa_rank\',\'confirmed80\',\'observed_peak3\']];currentRanking.forEach((p,i)=>rows.push([p.player_id,p.edition,p.short_name,p.overall,p.potential,p.position_group,i+1,fr.get(p.player_id),reveal?(p.event_quality80??\'\'):\'\',reveal?(p.observed_peak3??\'\'):\'\']));$(\'csvtext\').value=rows.map(r=>r.map(v=>\'"\'+String(v).replaceAll(\'"\',\'""\')+\'"\').join(\',\')).join(\'\\n\');$(\'csvnote\').textContent=currentRanking.length.toLocaleString()+\' players in this filtered edition. Select and copy the CSV text. Future fields follow the reveal setting.\';$(\'csvpanel\').classList.remove(\'hidden\');$(\'csvtext\').focus();$(\'csvtext\').select();}\n(async()=>{try{let bytes=Uint8Array.from(atob($(\'packed\').textContent.trim()),c=>c.charCodeAt(0));let stream=new Blob([bytes]).stream().pipeThrough(new DecompressionStream(\'gzip\'));DATA=JSON.parse(await new Response(stream).text());all=DATA.players.map(row=>Object.fromEntries(DATA.columns.map((c,i)=>[c,row[i]])));options(\'year\',DATA.years.map(y=>[y,y===2026?\'FC26 snapshot\':String(y)]),2023);options(\'role\',[[\'all\',\'All positions\'],...[\'CB\',\'FB\',\'MF\',\'Wing\',\'FW\',\'GK\'].map(r=>[r,r])]);options(\'country\',[[\'all\',\'All nationalities\'],...[...new Set(all.map(p=>p.nationality_name))].sort().map(c=>[c,c])]);options(\'model\',[[DATA.base,DATA.methods[DATA.base]],[DATA.fifa,DATA.methods[DATA.fifa]]],DATA.base);[\'year\',\'role\',\'country\',\'ceiling\',\'budget\',\'model\',\'colour\',\'reveal\',\'never70\'].forEach(id=>$(id).onchange=update);$(\'search\').oninput=drawTable;$(\'download\').onclick=exportCSV;$(\'loading\').classList.add(\'hidden\');$(\'room\').classList.remove(\'hidden\');update();}catch(e){$(\'loading\').textContent=\'This browser could not open the interactive replay. \'+e.message;}})();\n'

from pathlib import Path
import os
import numpy as np
import pandas as pd
INPUT_ROOT=Path(os.environ.get('FIFA_INPUT_ROOT','/kaggle/input' if Path('/kaggle/input').exists() else 'data'))
SKILLS = [
    'attacking_crossing','attacking_finishing','attacking_heading_accuracy','attacking_short_passing','attacking_volleys',
    'skill_dribbling','skill_curve','skill_fk_accuracy','skill_long_passing','skill_ball_control',
    'movement_acceleration','movement_sprint_speed','movement_agility','movement_reactions','movement_balance',
    'power_shot_power','power_jumping','power_stamina','power_strength','power_long_shots',
    'mentality_aggression','mentality_interceptions','mentality_positioning','mentality_vision','mentality_penalties','mentality_composure',
    'defending_marking_awareness','defending_standing_tackle','defending_sliding_tackle',
    'goalkeeping_diving','goalkeeping_handling','goalkeeping_kicking','goalkeeping_positioning','goalkeeping_reflexes',
]

NUMERIC = ['overall','potential','age','height_cm','weight_kg','weak_foot','skill_moves','international_reputation',
           'pace','shooting','passing','dribbling','defending','physic'] + SKILLS

def locate(name,hint):
    found=[p for p in INPUT_ROOT.rglob(name) if hint in str(p)]
    if len(found)!=1: raise FileNotFoundError(f'Attach {hint}: expected one {name}, found {len(found)} beneath {INPUT_ROOT}.')
    return found[0]

def load_ratings():
    base_path = locate('male_players.csv', 'ea-sports-fc-24-complete-player-dataset')
    fc25_path = locate('male_players.csv', 'ea-sports-fc-25-database-ratings-and-stats')
    fc26_path = locate('FC26_20250921.csv', 'fc-26-fifa-26-player-data')
    base = pd.read_csv(base_path, low_memory=False)
    base['edition'] = 2000 + pd.to_numeric(base['fifa_version']).astype(int)
    base = base.sort_values(['edition','fifa_update','player_id']).drop_duplicates(['player_id','edition'])
    fc26 = pd.read_csv(fc26_path, low_memory=False)
    fc26['edition'] = 2026
    fc25 = pd.read_csv(fc25_path, low_memory=False)
    names = {
        'Name':'short_name','OVR':'overall','PAC':'pace','SHO':'shooting','PAS':'passing','DRI':'dribbling','DEF':'defending','PHY':'physic',
        'Age':'age','Position':'player_positions','Weak foot':'weak_foot','Skill moves':'skill_moves','Preferred foot':'preferred_foot',
        'Team':'club_name','League':'league_name','Nation':'nationality_name',
        'Acceleration':'movement_acceleration','Sprint Speed':'movement_sprint_speed','Positioning':'mentality_positioning',
        'Finishing':'attacking_finishing','Shot Power':'power_shot_power','Long Shots':'power_long_shots','Volleys':'attacking_volleys',
        'Penalties':'mentality_penalties','Vision':'mentality_vision','Crossing':'attacking_crossing','Free Kick Accuracy':'skill_fk_accuracy',
        'Short Passing':'attacking_short_passing','Long Passing':'skill_long_passing','Curve':'skill_curve','Dribbling':'skill_dribbling',
        'Agility':'movement_agility','Balance':'movement_balance','Reactions':'movement_reactions','Ball Control':'skill_ball_control',
        'Composure':'mentality_composure','Interceptions':'mentality_interceptions','Heading Accuracy':'attacking_heading_accuracy',
        'Def Awareness':'defending_marking_awareness','Standing Tackle':'defending_standing_tackle','Sliding Tackle':'defending_sliding_tackle',
        'Jumping':'power_jumping','Stamina':'power_stamina','Strength':'power_strength','Aggression':'mentality_aggression',
        'GK Diving':'goalkeeping_diving','GK Handling':'goalkeeping_handling','GK Kicking':'goalkeeping_kicking',
        'GK Positioning':'goalkeeping_positioning','GK Reflexes':'goalkeeping_reflexes',
    }
    fc25 = fc25.rename(columns=names)
    fc25['player_id'] = pd.to_numeric(fc25['url'].str.extract(r'/([0-9]+)/?$')[0], errors='raise').astype(int)
    assert fc25.player_id.is_unique, 'Duplicate FC25 player IDs need review.'
    fc25['edition'] = 2025
    fc25['long_name'] = fc25.short_name
    fc25['height_cm'] = pd.to_numeric(fc25['Height'].str.extract(r'^(\d+)')[0], errors='coerce')
    fc25['weight_kg'] = pd.to_numeric(fc25['Weight'].str.extract(r'^(\d+)')[0], errors='coerce')
    keep = ['player_id','edition','short_name','long_name','player_positions','preferred_foot','club_name','league_name','nationality_name','club_team_id','league_id','league_level','club_position','club_joined_date','club_loaned_from','club_contract_valid_until_year'] + NUMERIC
    frames, audit = [], []
    for frame, source in [(base,'FIFA15–FC24 / SoFIFA'),(fc25,'FC25 / EA ratings'),(fc26,'FC26 / SoFIFA')]:
        for c in keep:
            if c not in frame:
                frame[c] = np.nan
        for c in NUMERIC:
            frame[c] = pd.to_numeric(frame[c], errors='coerce')
        frame['player_id'] = pd.to_numeric(frame.player_id, errors='raise').astype(int)
        frame['source'] = source
        for year, group in frame.groupby('edition'):
            audit.append({'edition':int(year),'players':len(group),'source':source,
                          'mean_skill_fields_present':group[SKILLS].notna().sum(axis=1).mean(),
                          'potential_present':group.potential.notna().mean()})
        frames.append(frame[list(dict.fromkeys(keep + ['source']))])
    panel = pd.concat(frames, ignore_index=True).sort_values(['player_id','edition']).reset_index(drop=True)
    assert not panel.duplicated(['player_id','edition']).any()
    assert panel.overall.between(1,99).all()
    assert sorted(panel.edition.unique()) == list(range(2015,2027)), 'The study needs all twelve editions, 2015–2026.'
    fc24_ids = set(base.loc[base.edition.eq(2024),'player_id'])
    common25 = set(fc25.player_id).intersection(fc24_ids)
    assert len(common25) / len(fc25) > 0.5, 'FC25 IDs do not align with historical FIFA IDs.'
    overlapping = base.loc[base.edition.eq(2024),['player_id','overall']].merge(fc25[['player_id','overall']],on='player_id')
    assert overlapping[['overall_x','overall_y']].corr().iloc[0,1] > 0.75, 'Check the cross-source identity bridge.'
    panel['primary_position'] = panel.player_positions.fillna('Unknown').str.split(',').str[0].str.strip()
    positions = {'GK':'GK','CB':'CB','LB':'FB','RB':'FB','LWB':'FB','RWB':'FB','CDM':'MF','CM':'MF','CAM':'MF',
                 'LM':'Wing','RM':'Wing','LW':'Wing','RW':'Wing','ST':'FW','CF':'FW'}
    panel['position_group'] = panel.primary_position.map(positions).fillna('Other')
    panel['row_id'] = np.arange(len(panel))
    return panel, pd.DataFrame(audit), [base_path,fc25_path,fc26_path]

ROLE_CORES={
    'FW':['attacking_finishing','mentality_positioning','skill_ball_control','movement_reactions','power_shot_power'],
    'Wing':['movement_acceleration','movement_sprint_speed','skill_dribbling','skill_ball_control','mentality_vision'],
    'MF':['attacking_short_passing','skill_long_passing','mentality_vision','skill_ball_control','movement_reactions'],
    'CB':['defending_marking_awareness','defending_standing_tackle','mentality_interceptions','power_strength','movement_reactions'],
    'FB':['power_stamina','attacking_crossing','movement_acceleration','movement_sprint_speed','defending_standing_tackle'],
    'GK':['goalkeeping_diving','goalkeeping_handling','goalkeeping_kicking','goalkeeping_positioning','goalkeeping_reflexes'],
    'Other':['skill_ball_control','movement_reactions','power_stamina'],
}

TECHNICAL=['attacking_short_passing','skill_ball_control','skill_dribbling','mentality_vision']

ATHLETIC=['movement_acceleration','movement_sprint_speed','movement_agility','power_stamina']

MENTAL=['movement_reactions','mentality_composure','mentality_positioning','mentality_vision']

import warnings
from types import SimpleNamespace
CFG=SimpleNamespace(min_age=16,max_age=99,training_max_age=99,rating_cap=69,star_rating=85)
def strong_player_engineering(raw):
    p=raw.sort_values(['player_id','edition']).reset_index(drop=True).copy()
    p['peak_seen']=p.groupby('player_id').overall.cummax()
    p['is_candidate']=p.age.between(CFG.min_age,CFG.max_age)&p.overall.le(CFG.rating_cap)&p.peak_seen.le(CFG.rating_cap)
    p['is_training']=p.age.between(CFG.min_age,CFG.training_max_age)&p.overall.lt(CFG.star_rating)&p.peak_seen.lt(CFG.star_rating)
    p['role_quality']=np.nan
    for role,cols in ROLE_CORES.items():
        mask=p.position_group.eq(role);p.loc[mask,'role_quality']=p.loc[mask,cols].mean(axis=1)
    for name,cols in [('technical',TECHNICAL),('athletic',ATHLETIC),('mental',MENTAL)]:
        p[name+'_quality']=p[cols].mean(axis=1)
        p[name+'_headroom']=p[name+'_quality']-p.overall
    p['role_headroom']=p.role_quality-p.overall
    p['skill_spikiness']=p[SKILLS].quantile(.9,axis=1)-p[SKILLS].median(axis=1)
    p['attributes_75plus']=p[SKILLS].ge(75).sum(axis=1)
    p['attributes_80plus']=p[SKILLS].ge(80).sum(axis=1)
    p['weakest_role_skill']=np.nan
    for role,cols in ROLE_CORES.items():
        mask=p.position_group.eq(role);p.loc[mask,'weakest_role_skill']=p.loc[mask,cols].min(axis=1)
    p['role_balance']=p.weakest_role_skill-p.role_quality
    p['age_to_21']=21-p.age
    p['age_rating_interaction']=(21-p.age)*p.overall/100
    for role in ROLE_CORES:p['role_'+role]=p.position_group.eq(role).astype(int)
    p['right_foot']=p.preferred_foot.str.lower().eq('right').astype(int)
    p['age_band']=np.floor(p.age/2)*2
    peers=p.groupby(['edition','position_group','age_band'],observed=True)
    for c in ['role_quality','technical_quality','athletic_quality','mental_quality']:
                                                                                
        p[c+'_peer_percentile']=peers[c].rank(pct=True)
    tracked=['overall','role_quality','technical_quality','athletic_quality','mental_quality',
             'movement_reactions','skill_ball_control','power_stamina']
    index=p.set_index(['player_id','edition'])
    for lag in [1,2]:
        keys=pd.MultiIndex.from_arrays([p.player_id,p.edition-lag],names=['player_id','edition'])
        values=index[tracked].reindex(keys)
        for c in tracked:
            p[c+f'_lag{lag}']=values[c].to_numpy()
            p[c+f'_change{lag}']=p[c]-p[c+f'_lag{lag}']
    p['history_length']=1+p.overall_lag1.notna().astype(int)+(p.overall_lag1.notna()&p.overall_lag2.notna()).astype(int)
    p['rating_acceleration']=p.overall_change1-(p.overall_lag1-p.overall_lag2)
    p['core_growth_ahead_of_rating']=p.role_quality_change1-p.overall_change1
    p['technical_growth_ahead_of_rating']=p.technical_quality_change1-p.overall_change1
    p['potential_gap']=p.potential-p.overall
    p['under_radar']=p.potential.lt(80)
    current=['overall','age','height_cm','weight_kg','weak_foot','skill_moves','pace','shooting','passing','dribbling','defending','physic']+SKILLS+['right_foot']+['role_'+r for r in ROLE_CORES]
    engineered=['role_quality','technical_quality','athletic_quality','mental_quality','role_headroom','technical_headroom',
        'athletic_headroom','mental_headroom','skill_spikiness','attributes_75plus','attributes_80plus','role_balance',
        'weakest_role_skill','age_to_21','age_rating_interaction']+[c+'_peer_percentile' for c in ['role_quality','technical_quality','athletic_quality','mental_quality']]
    history=[c+suffix for c in tracked for suffix in ['_lag1','_change1','_lag2','_change2']]+['history_length','rating_acceleration','core_growth_ahead_of_rating','technical_growth_ahead_of_rating']
    sets={'Snapshot skills':current,'Hidden-gem features':current+engineered+history,
          'Skills + EA potential':current+engineered+history+['potential','potential_gap']}
    for name,cols in sets.items():
        assert len(cols)==len(set(cols))
        assert not any(c.startswith('future') or c in ['player_id','edition','short_name','club_name','league_name'] for c in cols)
        if name!='Skills + EA potential':assert not any('potential' in c for c in cols)
    return p,sets

def strong_current_context(raw):
    p, old_sets = strong_player_engineering(raw)
                                                                                 
    p['eligible'] = p.overall.lt(70) & p.peak_seen.lt(85)
    p['training_pool'] = p.overall.lt(85) & p.peak_seen.lt(85)
    club = p.club_name.fillna('').astype(str).str.strip().str.casefold()
    league = p.league_name.fillna('').astype(str).str.strip().str.casefold()
    p['_club'] = club.where(club.ne(''), 'missing_player_' + p.player_id.astype(str))
    p['_league'] = league.where(league.ne(''), 'missing_league_' + p.player_id.astype(str))
    valid_club = club.ne('') & league.ne('')
    keys = ['edition', '_league', '_club']
    grouped = p.groupby(keys, observed=True, sort=False)
    teams = grouped.agg(team_roster_n=('overall', 'size'), team_mean_rating=('overall', 'mean'),
        team_median_rating=('overall', 'median'), team_rating_std=('overall', 'std'),
        team_best_rating=('overall', 'max'), team_mean_age=('age', 'mean'))
    top11 = p.sort_values('overall', ascending=False).groupby(keys, observed=True, sort=False).head(11)
    teams['team_top11_rating'] = top11.groupby(keys, observed=True).overall.mean()
    teams['team_u21_share'] = p.assign(_u21=p.age.le(21).astype(float)).groupby(keys, observed=True)._u21.mean()
    teams['team_80plus_count'] = p.assign(_elite=p.overall.ge(80).astype(float)).groupby(keys, observed=True)._elite.sum()
    teams = teams.reset_index()
    leagues = teams.groupby(['edition', '_league'], observed=True)
    teams['team_strength_league_percentile'] = leagues.team_top11_rating.rank(pct=True, method='average')
    teams['league_team_count'] = leagues.team_top11_rating.transform('size')
    teams['league_mean_team_strength'] = leagues.team_top11_rating.transform('mean')
    teams['league_best_team_strength'] = leagues.team_top11_rating.transform('max')
    teams['team_strength_vs_league'] = teams.team_top11_rating - teams.league_mean_team_strength
    teams['team_strength_vs_league_best'] = teams.team_top11_rating - teams.league_best_team_strength
    teams['team_strength_global_percentile'] = teams.groupby('edition').team_top11_rating.rank(pct=True)
    p = p.merge(teams, on=keys, how='left', validate='many_to_one', sort=False)
    p = p.sort_values(['player_id', 'edition']).reset_index(drop=True)
    context = list(teams.columns.difference(keys, sort=False))
    p['rating_vs_team_top11'] = p.overall - p.team_top11_rating
    p['rating_vs_team_mean'] = p.overall - p.team_mean_rating
    p['age_vs_team_mean'] = p.age - p.team_mean_age
    p['team_roster_log_n'] = np.log1p(p.team_roster_n)
    p['league_level_numeric'] = pd.to_numeric(p.league_level, errors='coerce')
    p['previous_peak_gap'] = p.peak_seen - p.overall
    p['first_observed_age'] = p.groupby('player_id').age.transform('first')
    p['editions_since_first_seen'] = p.edition - p.groupby('player_id').edition.transform('first')
    context += ['rating_vs_team_top11', 'rating_vs_team_mean', 'age_vs_team_mean', 'team_roster_log_n',
                'league_level_numeric', 'previous_peak_gap', 'first_observed_age', 'editions_since_first_seen']
    rating_fields = list(dict.fromkeys(['overall', 'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic',
        'role_quality', 'technical_quality', 'athletic_quality', 'mental_quality'] + SKILLS))
                                                                                      
    comparisons = [
        ('team', ['edition', '_league', '_club'], rating_fields),
        ('position', ['edition', 'primary_position'], rating_fields),
        ('position_age', ['edition', 'primary_position', 'age'], rating_fields),
        ('team_role', ['edition', '_league', '_club', 'position_group'],
            ['overall', 'role_quality', 'technical_quality', 'athletic_quality', 'mental_quality', 'movement_reactions']),
        ('league_position', ['edition', '_league', 'primary_position'],
            ['overall', 'role_quality', 'technical_quality', 'athletic_quality', 'mental_quality', 'pace']),
        ('league_position_age', ['edition', '_league', 'primary_position', 'age'],
            ['overall', 'role_quality', 'technical_quality', 'athletic_quality', 'mental_quality', 'pace']),
    ]
    for prefix, by, columns in comparisons:
        g = p.groupby(by, observed=True, sort=False)
        group_n = g.overall.transform('size')
        p[prefix + '_peer_log_n'] = np.log1p(group_n)
        context.append(prefix + '_peer_log_n')
        for c in columns:
            counts = g[c].transform('count')
            ranks = g[c].rank(method='average', pct=False)
                                                                                   
            percentile = (ranks - .5) / counts
            p[prefix + '_pct_' + c] = .5 + (percentile - .5) * counts / (counts + 5)
            context.append(prefix + '_pct_' + c)
            if prefix in ['team', 'team_role']:
                sums = g[c].transform('sum')
                peer_mean = (sums - p[c]) / (counts - p[c].notna().astype(int)).replace(0, np.nan)
                p[prefix + '_gap_' + c] = p[c] - peer_mean
                context.append(prefix + '_gap_' + c)
        if prefix == 'team_role':
            p['team_role_rating_depth_gap'] = p.overall - g.overall.transform('max')
            context.append('team_role_rating_depth_gap')
    p['team_skill_above_rating_rank'] = p.team_pct_role_quality - p.team_pct_overall
    p['position_age_skill_above_rating_rank'] = p.position_age_pct_role_quality - p.position_age_pct_overall
    context += ['team_skill_above_rating_rank', 'position_age_skill_above_rating_rank']
    lineup = p.club_position.fillna('').astype(str)
    for name, mask in [('named_starter', ~lineup.isin(['SUB', 'RES', ''])),
                       ('substitute', lineup.eq('SUB')), ('reserve', lineup.eq('RES'))]:
        p['club_' + name] = mask.astype(float).where(lineup.ne(''))
        context.append('club_' + name)
                                                               
    index = p.set_index(['player_id', 'edition'])
    tracking = ['team_top11_rating', 'team_strength_league_percentile', 'league_mean_team_strength',
        'rating_vs_team_mean', 'team_pct_overall', 'team_pct_role_quality', 'team_role_pct_role_quality',
        'position_age_pct_role_quality', 'team_role_rating_depth_gap']
    for lag in [1, 2]:
        lookup = pd.MultiIndex.from_arrays([p.player_id, p.edition-lag], names=['player_id', 'edition'])
        historical = index[tracking + ['club_team_id', 'league_id']].reindex(lookup)
        for c in tracking:
            p[c + f'_context_change{lag}'] = p[c].to_numpy() - historical[c].to_numpy()
            context.append(c + f'_context_change{lag}')
        for name in ['club_team_id', 'league_id']:
            now = pd.to_numeric(p[name], errors='coerce').to_numpy()
            then = pd.to_numeric(historical[name], errors='coerce').to_numpy()
            p[name + f'_changed{lag}'] = np.where(np.isfinite(now) & np.isfinite(then), (now != then).astype(float), np.nan)
            context.append(name + f'_changed{lag}')
    context = list(dict.fromkeys(context))
    club_features = [c for c in context if c.startswith(('team_', 'rating_vs_team', 'age_vs_team', 'club_', 'league_'))]
    p.loc[~valid_club.to_numpy(), club_features] = np.nan
    base_features = old_sets['Hidden-gem features']
    sets = {'Skills only': base_features, 'Skills + context': base_features + context,
            'Context + EA': base_features + context + ['potential', 'potential_gap']}
    for name, cols in sets.items():
        assert len(cols) == len(set(cols))
        assert not set(cols).intersection(['player_id', 'row_id', 'edition', 'club_name', 'league_name', 'short_name'])
        if name != 'Context + EA':
            assert not any('potential' in c for c in cols)
    p[list(dict.fromkeys(sum(sets.values(), [])))] = p[list(dict.fromkeys(sum(sets.values(), [])))].astype('float32')
    return p, sets, context

def strong_history_panel(raw):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', pd.errors.PerformanceWarning)
        p, old_sets, context = strong_current_context(raw)
    p = p.copy()
    index = p.set_index(['player_id', 'edition'])
    new = {}
    tracked = ['overall', 'role_quality', 'technical_quality', 'athletic_quality', 'mental_quality',
               'team_top11_rating', 'movement_reactions', 'skill_ball_control', 'power_stamina']
    histories = {c: [p[c].to_numpy()] for c in tracked}
    for lag in range(1, 5):
        keys = pd.MultiIndex.from_arrays([p.player_id, p.edition-lag], names=['player_id', 'edition'])
        past = index[tracked].reindex(keys)
        for c in tracked:
            values = past[c].to_numpy()
            histories[c].append(values)
            if lag >= 3:
                new[c+f'_lag{lag}'] = values
                new[c+f'_change{lag}'] = p[c].to_numpy()-values
    for c, arrays in histories.items():
        values = np.column_stack(arrays)
        finite = np.isfinite(values)
        n = finite.sum(axis=1)
        t = -np.arange(5, dtype=float)[None, :]
        sx = np.where(finite, t, 0).sum(axis=1)
        sy = np.where(finite, values, 0).sum(axis=1)
        sxx = np.where(finite, t*t, 0).sum(axis=1)
        sxy = np.where(finite, t*values, 0).sum(axis=1)
        denominator = n*sxx-sx*sx
        new[c+'_past_slope'] = np.divide(n*sxy-sx*sy, denominator,
            out=np.full(len(p), np.nan), where=denominator>0)
        new[c+'_past_observations'] = n
    ratings = np.column_stack(histories['overall'])
    new['rating_below_recent_peak'] = np.nanmax(ratings, axis=1)-p.overall.to_numpy()
    new['rating_above_recent_floor'] = p.overall.to_numpy()-np.nanmin(ratings, axis=1)
    new['past_growth_skill_ahead'] = new['role_quality_past_slope']-new['overall_past_slope']
    new['recent_growth_consistency'] = np.divide(
        (np.diff(ratings, axis=1)<0).sum(axis=1),
        np.isfinite(np.diff(ratings, axis=1)).sum(axis=1),
        out=np.full(len(p), np.nan), where=np.isfinite(np.diff(ratings, axis=1)).sum(axis=1)>0)
    p = pd.concat([p, pd.DataFrame(new, index=p.index).astype('float32')], axis=1)
    p['eligible'] = p.overall.lt(70) & p.peak_seen.lt(85)
    p['fit_domain'] = p.overall.lt(80) & p.peak_seen.lt(85)
    p['country'] = p.nationality_name.fillna('Unknown').astype(str)
    for h in range(1, 4):
        keys = pd.MultiIndex.from_arrays([p.player_id, p.edition+h], names=['player_id', 'edition'])
        p[f'next_rating_{h}'] = index.overall.reindex(keys).to_numpy()
    future = p[[f'next_rating_{h}' for h in range(1,4)]]
    p['observed_peak3'] = future.max(axis=1)
    p['complete3'] = future.notna().all(axis=1)
    p['delta3_observed'] = p.observed_peak3-p.overall
    p['delta3'] = p.delta3_observed.where(p.complete3)
    p['outcome_end'] = p.edition+3
    sets = {'Pattern': old_sets['Skills + context']+list(new),
            'Pattern + FIFA': old_sets['Context + EA']+list(new)}
    for name, cols in sets.items():
        assert len(cols)==len(set(cols))
        assert not any(c.startswith(('future_', 'next_', 'delta', 'observed_peak')) for c in cols)
        assert not set(cols).intersection(['player_id','row_id','edition','country','nationality_name','short_name','club_name'])
        if name=='Pattern': assert not any('potential' in c for c in cols)
    return p, sets, context+list(new)

from sklearn.decomposition import PCA
import joblib,json,warnings

ROLES=['CB','FB','MF','Wing','FW']
LEAGUES=[13,16,19,31,53]
ROLE_LABELS={'CB':'Centre-backs','FB':'Full-backs','MF':'Central midfielders','Wing':'Wide players','FW':'Forwards','GK':'Goalkeepers'}
OUT=Path(os.environ.get('FUTURE_OUTPUT','results'));OUT.mkdir(exist_ok=True,parents=True)
SEED=42

def canonical_ids(raw):
    p=raw.copy()
    original=p[['row_id','club_name','league_name']].copy()
    native=p.club_team_id.notna()&p.league_id.notna()
    p.loc[native,'club_name']='club_'+p.loc[native,'club_team_id'].astype(int).astype(str)
    p.loc[native,'league_name']='league_'+p.loc[native,'league_id'].astype(int).astype(str)
    p.loc[~native&p.club_name.isna(),'club_name']=''
    p.loc[~native&p.league_name.isna(),'league_name']=''
    audit=raw.groupby(['edition','league_id']).league_name.nunique().reset_index(name='name_variants')
    audit=audit.loc[audit.name_variants.gt(1)]
    return p,original,audit

def normalize_shapes(raw):
                                                                                  
    seed=raw.loc[raw.edition.eq(2015)&raw.position_group.isin(ROLES)]
    skills=[c for c in SKILLS if not c.startswith('goalkeeping_') and seed[c].notna().mean()>.95]
    def design(f):
        a=(f.age.to_numpy(float)-25)/10;o=(f.overall.to_numpy(float)-70)/10
        return np.column_stack([np.ones(len(f)),a,a*a,o,o*o]+[f.position_group.eq(r).to_numpy(float) for r in ROLES])
    med=seed[skills].median().to_numpy();x=design(seed);y=seed[skills].fillna(dict(zip(skills,med))).to_numpy(float)
    penalty=np.eye(x.shape[1])*20;penalty[0,0]=0
    beta=np.linalg.solve(x.T@x+penalty,x.T@y)
    resid=y-x@beta;scale=np.maximum(resid.std(axis=0),3.)
    def transform(f):
        z=(f[skills].fillna(dict(zip(skills,med))).to_numpy(float)-design(f)@beta)/scale
        z-=z.mean(axis=1,keepdims=True)
        return np.clip(z,-5,5)
    z=transform(raw)
    pc=PCA(n_components=2,svd_solver='full').fit(transform(seed))
    coordinates=pc.transform(z)
    bundle={'skills':skills,'medians':med,'beta':beta,'scale':scale,'pca':pc,'normalizer_fit_end':2015,
            'explained_variance':pc.explained_variance_ratio_.tolist()}
    return z,coordinates,bundle

def cohort_profiles(raw,z,skills):
    f=raw.copy();f['_league_id']=f.league_id
    aliases={'Premier League':13,'Ligue 1 McDonald\'s':16,'Bundesliga':19,'Serie A Enilive':31,'LALIGA EA SPORTS':53}
    missing=f._league_id.isna();f.loc[missing,'_league_id']=f.loc[missing,'league_name'].map(aliases)
                                                                                                       
    f['_age_bin']=np.where(f.age.le(25),0,1)
    f['_band']=np.select([f.overall.between(80,89),f.overall.between(70,79)],['elite','reference'],default='omit')
    mask=f.age.between(20,32)&f.position_group.isin(ROLES)&f._league_id.isin(LEAGUES)&f._band.ne('omit')
    rows=f.loc[mask,['player_id','edition','position_group','_league_id','_age_bin','_band']].copy()
    zz=['z'+str(i) for i in range(z.shape[1])]
    rows[zz]=z[mask]
    keys=['edition','position_group','_league_id','_age_bin','_band']
    grouped=rows.groupby(keys)[zz].mean();n=rows.groupby(keys).size()
    records=[];profiles={};support={}
                                                                                                   
    for (year,role),part in rows.groupby(['edition','position_group']):
        cells=[];counts=[]
        for league in LEAGUES:
            for age in [0,1]:
                ke=(year,role,league,age,'elite');kr=(year,role,league,age,'reference')
                if ke in n and kr in n and n.loc[ke]>=2 and n.loc[kr]>=2:
                    cells.append(grouped.loc[ke].to_numpy()-grouped.loc[kr].to_numpy())
                    counts.append(int(n.loc[ke]+n.loc[kr]))
        if len(cells)<3:continue
        mu=np.mean(cells,axis=0);profiles[(int(year),role)]=mu
        support[(int(year),role)]={'cells':len(cells),'profiles':sum(counts)}
        for skill,value in zip(skills,mu):records.append({'edition':int(year),'role':role,'skill':skill,'elite_premium':float(value),'cells':len(cells),'profiles':sum(counts)})
    return profiles,support,pd.DataFrame(records),rows

STATIC=['role_fit_now','role_distance_now','role_fit_available','role_reference_cells']
DYNAMIC=['role_fit_forecast','role_distance_forecast','future_fit_advantage','trend_alignment','role_trend_speed','role_trend_coherence','role_trend_years','role_trend_available']
LEARNED=['learned_future_fit','learned_future_distance','learned_future_advantage','learned_future_alignment',
         'learned_change_size','learned_forecast_training_r2','learned_forecast_transitions','learned_forecast_available']

def learn_profile_forecasts(raw,z,profiles):
    values={c:np.full(len(raw),np.nan) for c in LEARNED};values[LEARNED[-1]][:]=0
    audit=[];scores=[];coefficients=[]
    def predictors(h):
        h=np.asarray(h[-4:]);t=np.arange(len(h),dtype=float);t-=t.mean()
        slope=(t[:,None]*(h-h.mean(axis=0))).sum(axis=0)/max(t@t,1e-8)
        return np.column_stack([np.mean(h[-3:],axis=0)-h[-1],slope,h[-1]-h[-2]])
    for year in sorted(raw.edition.unique()):
        xx=[];yy=[];origins=[]
        for r in ROLES:
            for origin in range(2016,int(year)):
                past=[profiles[(s,r)] for s in range(max(2015,origin-3),origin+1) if (s,r) in profiles]
                if len(past)<2 or (origin+1,r) not in profiles:continue
                xx.append(predictors(past));yy.append(profiles[(origin+1,r)]-past[-1]);origins.append(origin)
        if not xx:continue
        x=np.vstack(xx);y=np.concatenate(yy);scale=np.maximum(np.std(x,axis=0),.01);xs=x/scale
        coef=np.linalg.solve(xs.T@xs+20*np.eye(3),xs.T@y)/scale
        fitted=x@coef;r2=1-np.sum((y-fitted)**2)/max(np.sum(y*y),1e-8)
        coefficients.append({'origin':int(year),'toward_recent_mean':coef[0],'past_slope':coef[1],'last_change':coef[2],
                             'last_training_profile':int(max(origins)+1),'past_role_transitions':len(origins),'training_r2':r2})
        for r in ROLES:
            if (int(year),r) not in profiles:continue
            h=[profiles[(s,r)] for s in range(max(2015,int(year)-3),int(year)+1) if (s,r) in profiles]
            if len(h)<2:continue
            mu=h[-1].copy();forecast_history=[a.copy() for a in h]
            for _ in range(3):
                step=predictors(forecast_history)@coef
                forecast_history.append(forecast_history[-1]+np.clip(step,-.25,.25))
            future=forecast_history[-1];change=future-mu
            loc=np.flatnonzero(raw.edition.eq(year)&raw.position_group.eq(r));a=z[loc];an=np.maximum(np.linalg.norm(a,axis=1),1e-8)
            distance=np.sqrt(np.mean((a-mu)**2,axis=1));fd=np.sqrt(np.mean((a-future)**2,axis=1))
            vals=[(a@future)/(an*max(np.linalg.norm(future),1e-8)),fd,distance-fd,
                  (a@change)/(an*max(np.linalg.norm(change),1e-8)),np.sqrt(np.mean(change*change)),
                  r2,len(origins),1.]
            for c,v in zip(LEARNED,vals):values[c][loc]=v
            audit.append({'origin':int(year),'role':r,'last_training_profile':int(max(origins)+1),'first_training_origin':int(min(origins)),
                          'last_training_origin':int(max(origins)),'forecast_edition':int(year)+3})
            if (int(year)+3,r) in profiles:
                actual=profiles[(int(year)+3,r)];a0=np.sqrt(np.mean((actual-mu)**2));a1=np.sqrt(np.mean((actual-future)**2))
                scores.append({'origin':int(year),'role':r,'target_edition':int(year)+3,'static_rmse':a0,'trend_rmse':a1,'improvement':a0-a1})
    return pd.DataFrame(values,index=raw.index).astype('float32'),pd.DataFrame(audit),pd.DataFrame(scores),pd.DataFrame(coefficients)

def add_role_features(raw,z,profiles,support):
    values={c:np.full(len(raw),np.nan) for c in STATIC+DYNAMIC}
    values['role_fit_available'][:]=0;values['role_trend_available'][:]=0
    audit=[];forecasts=[]
    for (year,role),mu in profiles.items():
        loc=np.flatnonzero(raw.edition.eq(year)&raw.position_group.eq(role));a=z[loc];an=np.linalg.norm(a,axis=1)
        distance=np.sqrt(np.mean((a-mu)**2,axis=1))
        similarity=(a@mu)/(np.maximum(an,1e-8)*max(np.linalg.norm(mu),1e-8))
        values['role_fit_now'][loc]=similarity;values['role_distance_now'][loc]=distance
        values['role_fit_available'][loc]=1;values['role_reference_cells'][loc]=support[(year,role)]['cells']
        years=[y for y in range(max(2015,year-3),year+1) if (y,role) in profiles]
        if len(years)<3:continue
        history=np.vstack([profiles[(y,role)] for y in years]);t=np.asarray(years,float);t-=t.mean()
        slope=(t[:,None]*(history-history.mean(axis=0))).sum(axis=0)/(t@t)
        predicted=history.mean(axis=0)+t[:,None]*slope
        coherence=1-np.sum((history-predicted)**2)/max(np.sum((history-history.mean(axis=0))**2),1e-8)
                                                                                                  
        forecast=mu+1.5*slope
        fd=np.sqrt(np.mean((a-forecast)**2,axis=1));fn=max(np.linalg.norm(forecast),1e-8)
        values['role_fit_forecast'][loc]=(a@forecast)/(np.maximum(an,1e-8)*fn)
        values['role_distance_forecast'][loc]=fd
        values['future_fit_advantage'][loc]=distance-fd
        values['trend_alignment'][loc]=(a@slope)/(np.maximum(an,1e-8)*max(np.linalg.norm(slope),1e-8))
        values['role_trend_speed'][loc]=np.linalg.norm(slope)/np.sqrt(z.shape[1])
        values['role_trend_coherence'][loc]=coherence;values['role_trend_years'][loc]=len(years);values['role_trend_available'][loc]=1
        audit.append({'origin':year,'role':role,'first_profile':min(years),'last_profile':max(years),'profile_years':len(years),'forecast_edition':year+3})
        if (year+3,role) in profiles:
            actual=profiles[(year+3,role)]
            static_err=np.sqrt(np.mean((actual-mu)**2));future_err=np.sqrt(np.mean((actual-forecast)**2))
            forecasts.append({'origin':year,'role':role,'target_edition':year+3,'static_rmse':static_err,'trend_rmse':future_err,'improvement':static_err-future_err})
    return pd.DataFrame(values,index=raw.index).astype('float32'),pd.DataFrame(audit),pd.DataFrame(forecasts)

def build_role_panel(raw):
    z,coords,normalizer=normalize_shapes(raw)
    profiles,support,history,cohorts=cohort_profiles(raw,z,normalizer['skills'])
    features,audit,forecast=add_role_features(raw,z,profiles,support)
    learned,learned_audit,learned_scores,coefficients=learn_profile_forecasts(raw,z,profiles)
    features=pd.concat([features,learned],axis=1)
    return {'z':z,'coordinates':coords,'normalizer':normalizer,'profiles':profiles,'support':support,'profile_history':history,
            'cohort_rows':cohorts,'features':features,'trend_audit':audit,'profile_forecasts':forecast,
            'learned_audit':learned_audit,'learned_forecasts':learned_scores,'forecast_coefficients':coefficients}

def validate_time(raw,role):
    truncated=raw.loc[raw.edition.le(2020)].reset_index(drop=True)
    small=build_role_panel(truncated)
    ids=raw.loc[raw.edition.eq(2020),'row_id']
    a=role['features'].loc[raw.edition.eq(2020)].set_axis(ids.to_numpy())
    b=small['features'].loc[truncated.edition.eq(2020)].set_axis(truncated.loc[truncated.edition.eq(2020),'row_id'].to_numpy())
    pd.testing.assert_frame_equal(a.sort_index(),b.sort_index(),check_dtype=False,atol=1e-6,rtol=1e-6)
    assert role['trend_audit'].last_profile.le(role['trend_audit'].origin).all()
    assert role['normalizer']['normalizer_fit_end']==2015
    assert role['learned_audit'].last_training_profile.le(role['learned_audit'].origin).all()
    return {'future_removal_invariance':True,'normalizer_and_geometry_fit_2015':True,'role_profiles_known_at_origin':True,
            'candidates_excluded_from_prototypes':True}

def profile_shift_bootstrap(role,draws=400):
    f=role['cohort_rows'];f=f.loc[f.edition.isin([2015,2023])].copy();cols=[c for c in f if c.startswith('z')]
    f['_cell']=f._league_id.astype(int).astype(str)+'_'+f._age_bin.astype(str)
    output=[];rng=np.random.default_rng(79)
    for r in ROLES:
        g=f.loc[f.position_group.eq(r)].copy();groups=[(y,b,c) for y in [2015,2023] for b in ['elite','reference'] for c in g._cell.unique()]
        common=[c for c in g._cell.unique() if all(len(g.loc[g.edition.eq(y)&g._band.eq(b)&g._cell.eq(c)])>=2 for y in [2015,2023] for b in ['elite','reference'])]
        if not common:continue
        ids,iv=np.unique(g.player_id,return_inverse=True);arrays=g[cols].to_numpy();masks={(y,b,c):np.flatnonzero(g.edition.eq(y)&g._band.eq(b)&g._cell.eq(c)) for y,b,c in groups if c in common}
        def difference(w):
            means={}
            for y in [2015,2023]:
                pairs=[]
                for c in common:
                    a=masks[(y,'elite',c)];b=masks[(y,'reference',c)]
                    if w[a].sum()==0 or w[b].sum()==0:continue
                    pairs.append(np.average(arrays[a],axis=0,weights=w[a])-np.average(arrays[b],axis=0,weights=w[b]))
                means[y]=np.mean(pairs,axis=0)
            return means[2023]-means[2015]
        observed=difference(np.ones(len(g)));boot=[]
        for _ in range(draws):
            w=rng.multinomial(len(ids),np.full(len(ids),1/len(ids)))[iv];boot.append(difference(w))
        boot=np.asarray(boot);se=np.std(boot,axis=0,ddof=1)
        from scipy.stats import norm
        for j,skill in enumerate(role['normalizer']['skills']):
            output.append({'role':r,'skill':skill,'shift':observed[j],'low':np.quantile(boot[:,j],.025),'high':np.quantile(boot[:,j],.975),
                           'p_approx':2*norm.sf(abs(observed[j])/max(se[j],1e-8)),'matched_cells':len(common),'players':len(ids)})
    out=pd.DataFrame(output);p=out.p_approx.to_numpy();order=np.argsort(p);q=np.empty(len(p));q[order]=np.minimum.accumulate((p[order]*len(p)/np.arange(1,len(p)+1))[::-1])[::-1];out['q_bh']=np.minimum(q,1.)
    return out

from catboost import CatBoostClassifier,Pool
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score
import time,hashlib,importlib.metadata

YEARS=[2020,2021,2022,2023]
BASE='Player model';STATIC_MODEL='Player + current role';LINEAR='Player + linear trend';NEW='Player + learned future';FIFA='FIFA potential';CAL='Learned FIFA control'
METHODS=[FIFA,CAL,BASE,STATIC_MODEL,LINEAR,NEW]
CB_PARAMS=dict(iterations=450,depth=5,learning_rate=.045,l2_leaf_reg=12,border_count=64,
    loss_function='Logloss',random_seed=42,thread_count=4,verbose=False,allow_writing_files=False)

def model_frame(f,cols):return f[cols].astype('float32').fillna(-999.)

def make_panel(raw,role):
    clean,original,metadata_audit=canonical_ids(raw)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',pd.errors.PerformanceWarning)
        panel,sets,_=strong_history_panel(clean)
    lookup=raw.set_index('row_id')
    for c in ['short_name','club_name','league_name','nationality_name']:
        panel[c]=panel.row_id.map(lookup[c])
    feat=role['features'].copy();feat.index=raw.row_id
    for c in STATIC+DYNAMIC+LEARNED:panel[c]=panel.row_id.map(feat[c])
    for name,positive in [('quality80',panel.observed_peak3.ge(80)),('jump',panel.delta3_observed.ge(11)),('star85',panel.observed_peak3.ge(85))]:
        panel['event_'+name]=np.where(positive,1.,np.where(panel.complete3,0.,np.nan))
    cols=sets['Pattern + FIFA']
    variants={BASE:cols,STATIC_MODEL:cols+STATIC,LINEAR:cols+STATIC+DYNAMIC,NEW:cols+STATIC+LEARNED}
    assert len(cols)==431
    return panel,variants,metadata_audit

def dated_weights(train):
    counts=train.groupby('player_id').size()
    w=np.where(train.overall.lt(70),3.,1.)/train.player_id.map(counts).to_numpy()
    y=train.event_quality80.to_numpy()
    extra=1+np.clip((train.delta3_observed.to_numpy()-11)/10,0,2)
    w*=np.where(y==1,4*extra,1.)
    return w/np.mean(w)

def ordered(f,method):
    if method==FIFA:return f.sort_values([method,'overall','age','player_id'],ascending=[False,False,True,True],kind='mergesort')
    return f.sort_values([method,'player_id'],ascending=[False,True],kind='mergesort')

def measure(f,method,k=50,target='quality80'):
    r=ordered(f,method);g=r.head(min(k,len(r)));event='event_'+target
    found=int(g[event].eq(1).sum());unknown=int(g[event].isna().sum());total=int(r[event].eq(1).sum())
    return {'year':int(f.edition.iloc[0]),'method':method,'target':target,'k':k,'eligible':len(r),
            'confirmed_total':total,'found':found,'selected':len(g),'unresolved':unknown,
            'precision_lower':found/max(len(g),1),'precision_upper':(found+unknown)/max(len(g),1),
            'confirmed_recall':found/total if total else np.nan}

def fit_models(panel,variants,years=YEARS,reuse=False):
    outputs=[];audits=[];bundles={};tick=time.monotonic()
    for year in years:
        cache=OUT/f'fit_{year}.joblib'
        saved=None
        if reuse and cache.exists():
            out,bb,aa=joblib.load(cache)
            if 'Player + future role' in out:
                out=out.rename(columns={'Player + future role':LINEAR});bb[LINEAR]=bb.pop('Player + future role')
                for item in aa:
                    if item['method']=='Player + future role':item['method']=LINEAR
            if all(name in bb for name in variants):
                if 'peak_seen' not in out:out['peak_seen']=out.row_id.map(panel.set_index('row_id').peak_seen)
                outputs.append(out);audits+=aa;bundles[year]=bb;continue
            saved=(out,bb,aa)
        train=panel.loc[panel.fit_domain&panel.outcome_end.le(year)&panel.event_quality80.notna()].copy()
        test=panel.loc[panel.eligible&panel.edition.eq(year)&panel.potential.notna()].copy()
        assert len(test)>0 and train.outcome_end.max()<=year
        w=dated_weights(train);y=train.event_quality80.astype(int)
        meta=['row_id','player_id','edition','short_name','age','overall','potential','position_group','primary_position','nationality_name',
              'club_name','league_name','club_team_id','league_id','peak_seen','complete3','observed_peak3','delta3_observed','event_quality80','event_jump','event_star85']+STATIC+DYNAMIC+LEARNED
        out=test[meta].copy();out[FIFA]=out.potential;bb={};aa=[]
        if saved:
            old,bb,aa=saved
            assert np.array_equal(old.row_id,out.row_id)
            for name in list(bb)+[CAL]:
                if name in old:out[name]=old[name].to_numpy()
        baseline_features=['overall','age','potential','potential_gap']+['role_'+r for r in ROLE_CORES]
        control=LGBMClassifier(n_estimators=220,learning_rate=.04,num_leaves=7,max_depth=3,min_child_samples=100,
            reg_lambda=20,reg_alpha=1,n_jobs=4,verbosity=-1,random_state=42,deterministic=True,force_col_wise=True)
        if CAL not in out:
            control.fit(model_frame(train,baseline_features),y,sample_weight=w)
            out[CAL]=control.predict(model_frame(test,baseline_features),raw_score=True)
            bb[CAL]={'model':control,'columns':baseline_features}
        for name,cols in variants.items():
            if name in bb:continue
            model=CatBoostClassifier(**CB_PARAMS)
            model.fit(model_frame(train,cols),y,sample_weight=w)
            out[name]=model.predict(model_frame(test,cols),prediction_type='RawFormulaVal')
            bb[name]={'model':model,'columns':cols}
            aa.append({'year':year,'method':name,'features':len(cols),'training_rows':len(train),'positives':int(y.sum()),
                       'first_origin':int(train.edition.min()),'last_origin':int(train.edition.max()),'last_label':int(train.outcome_end.max())})
        joblib.dump((out,bb,aa),cache,compress=3);out.to_csv(OUT/f'forecasts_{year}.csv',index=False)
        outputs.append(out);audits+=aa;bundles[year]=bb
        print(f'{year}: {len(train):,} dated profiles, {int(y.sum()):,} positives; {time.monotonic()-tick:.1f}s',flush=True)
    forecasts=pd.concat(outputs,ignore_index=True);audit=pd.DataFrame(audits)
    forecasts.to_csv(OUT/'all_rankings.csv',index=False);audit.to_csv(OUT/'training_audit.csv',index=False)
    metrics=pd.DataFrame([measure(f,m,k,t) for year,f in forecasts.groupby('edition') if year in YEARS
                         for m in METHODS for k in [10,25,50,100,200,500] for t in ['quality80','jump','star85']])
    metrics.to_csv(OUT/'performance.csv',index=False)
    return forecasts,metrics,bundles,audit

def weighted_hit(order,event,weight,k):
    w=weight[order];before=np.cumsum(w)-w
    included=np.minimum(w,np.maximum(0,k-before))
    return float((included*event[order]).sum())

def uncertainty(forecasts,draws=500):
    f=forecasts.loc[forecasts.edition.isin(YEARS)].copy();ids,iv=np.unique(f.player_id,return_inverse=True)
    rng=np.random.default_rng(301);index=f.index.to_numpy();local={v:i for i,v in enumerate(index)}
    arrays={}
    for year,g in f.groupby('edition'):
        loc=np.array([local[i] for i in g.index]);orders={m:np.argsort(-g[m].to_numpy(),kind='stable') for m in METHODS}
                                                                                 
        orders={m:np.array([g.index.get_loc(i) for i in ordered(g,m).index]) for m in METHODS}
        event=g.event_quality80.eq(1).to_numpy(float);arrays[year]=(loc,orders,event)
    pairs=[(NEW,BASE),(NEW,STATIC_MODEL),(NEW,FIFA),(NEW,CAL),(BASE,FIFA)]
    observed={(year,a,b):measure(g,a)['found']-measure(g,b)['found'] for year,g in f.groupby('edition') for a,b in pairs}
    draws_by={key:[] for key in observed}
    for _ in range(draws):
        weights=rng.multinomial(len(ids),np.full(len(ids),1/len(ids)))[iv]
        for year,(loc,orders,event) in arrays.items():
            hits={m:weighted_hit(o,event,weights[loc],50) for m,o in orders.items()}
            for a,b in pairs:draws_by[(year,a,b)].append(hits[a]-hits[b])
    output=[]
    for a,b in pairs:
        total=np.sum([draws_by[(year,a,b)] for year in YEARS],axis=0)
        for year in YEARS:
            arr=draws_by[(year,a,b)];output.append({'year':year,'model':a,'baseline':b,'extra_hits':observed[(year,a,b)],
                'low':np.quantile(arr,.025),'high':np.quantile(arr,.975),'draws':draws,'scope':'Fixed models; player IDs resampled jointly across four editions; full rankings reranked at 50.'})
        output.append({'year':'Pooled','model':a,'baseline':b,'extra_hits':sum(observed[(year,a,b)] for year in YEARS),
                       'low':np.quantile(total,.025),'high':np.quantile(total,.975),'draws':draws,
                       'scope':'Conditional on these four cohorts, fitted models and observed records; not uncertainty over future football eras.'})
    out=pd.DataFrame(output);out.to_csv(OUT/'paired_player_bootstrap.csv',index=False);return out

def named_cases(panel,forecasts,role,bundles):
    f=forecasts.loc[forecasts.edition.eq(2023)].copy()
    for m,c in [(BASE,'ml_rank'),(NEW,'future_rank'),(FIFA,'fifa_rank')]:
        r=ordered(f,m);f[c]=r.index.to_series().map(pd.Series(np.arange(1,len(r)+1),index=r.index)).reindex(f.index)
                                                                                              
    confirmed=f.loc[f.event_quality80.eq(1)].copy();added=confirmed.loc[confirmed.ml_rank.le(50)&confirmed.fifa_rank.gt(50)].sort_values('ml_rank')
    if len(added)<2:added=pd.concat([added,confirmed.sort_values('ml_rank')]).drop_duplicates('player_id')
    pieces=[]
    if len(added):pieces.append(added.head(2).assign(case='Confirmed 80+'))
    false=f.loc[f.complete3&f.event_quality80.eq(0)].sort_values('ml_rank').head(1).assign(case='False alarm')
    if len(false):pieces.append(false)
    missed=confirmed.loc[confirmed.ml_rank.gt(50)].sort_values('fifa_rank').head(1).assign(case='Missed 80+')
    if len(missed):pieces.append(missed)
    cases=pd.concat(pieces).drop_duplicates('player_id').head(4)
    model=bundles[2023][BASE];profile=panel.set_index('row_id').reindex(cases.row_id)
    sv=model['model'].get_feature_importance(Pool(model_frame(profile,model['columns'])),type='ShapValues')
    raw=model['model'].predict(model_frame(profile,model['columns']),prediction_type='RawFormulaVal')
    np.testing.assert_allclose(sv[:,:-1].sum(axis=1)+sv[:,-1],raw,atol=1e-6)
    contrib=[]
    for i,(_,r) in enumerate(cases.iterrows()):
        for c,value in zip(model['columns'],sv[i,:-1]):
            group=feature_group(c)
            contrib.append({'player_id':int(r.player_id),'short_name':r.short_name,'feature':c,'shap':float(value),'group':group})
    neighbors=[];z=role['z'];raw_index=panel.set_index('row_id')
    meta=forecasts.loc[forecasts.edition.eq(2023)]
    for _,r in cases.iterrows():
        pool=panel.loc[panel.edition.eq(2023)&panel.position_group.eq(r.position_group)&panel.player_id.ne(r.player_id)&panel.overall.ge(75)]
        if r.position_group=='GK':
            gcols=[c for c in SKILLS if c.startswith('goalkeeping_')]
            seed=panel.loc[panel.edition.eq(2015)&panel.position_group.eq('GK'),gcols]
            scale=(seed.sub(seed.mean(axis=1),axis=0)).std().clip(lower=3).to_numpy()
            values=pool[gcols].to_numpy();values=(values-values.mean(axis=1,keepdims=True))/scale
            focal=panel.loc[panel.row_id.eq(r.row_id),gcols].to_numpy()[0];focal=(focal-focal.mean())/scale
            ds=np.sqrt(np.mean((values-focal)**2,axis=1));dimensions=5
        else:
            ds=np.sqrt(np.mean((z[pool.row_id.to_numpy()]-z[int(r.row_id)])**2,axis=1));dimensions=28
        for j in np.argsort(ds)[:4]:
            n=pool.iloc[j];neighbors.append({'player_id':int(r.player_id),'short_name':r.short_name,'neighbor_id':int(n.player_id),'neighbor':n.short_name,
                                           'neighbor_overall':int(n.overall),'distance':float(ds[j]),'edition':2023,'role':r.position_group,'dimensions':dimensions})
                                                                                
    subset=panel.loc[panel.edition.eq(2023)&panel.eligible].sample(min(1200,int((panel.edition.eq(2023)&panel.eligible).sum())),random_state=SEED)
    allsv=model['model'].get_feature_importance(Pool(model_frame(subset,model['columns'])),type='ShapValues')[:,:-1]
    importance=pd.DataFrame({'feature':model['columns'],'mean_abs_shap':np.abs(allsv).mean(axis=0),'method':BASE})
    other=bundles[2023][NEW]
    newshap=other['model'].get_feature_importance(Pool(model_frame(subset,other['columns'])),type='ShapValues')[:,:-1]
    importance=pd.concat([importance,pd.DataFrame({'feature':other['columns'],'mean_abs_shap':np.abs(newshap).mean(axis=0),'method':NEW})],ignore_index=True)
    cases.to_csv(OUT/'player_cases.csv',index=False);pd.DataFrame(contrib).to_csv(OUT/'player_shap.csv',index=False)
    pd.DataFrame(neighbors).to_csv(OUT/'player_neighbors.csv',index=False);importance.to_csv(OUT/'shap_importance.csv',index=False)
    return cases,pd.DataFrame(contrib),pd.DataFrame(neighbors),importance

def feature_group(c):
    if c in LEARNED:return 'Future role'
    if c in STATIC:return 'Current role'
    if c in ['potential','potential_gap']:return 'FIFA potential'
    if any(x in c for x in ['lag','change','past_','history','slope','acceleration','growth_consistency','recent_peak','recent_floor']):return 'Past development'
    if c.startswith(('team_','league_','club_','position_','rating_vs_','age_vs_')):return 'Team / peer context'
    if c in ['age','height_cm','weight_kg','age_to_21','age_rating_interaction','first_observed_age','editions_since_first_seen']:return 'Age / physique'
    return 'Current attributes'

def execute_future_study(reuse=True):
    source=OUT/'normalized_ratings.joblib'
    if source.exists():raw=joblib.load(source);source_audit=pd.DataFrame()
    else:
        raw,source_audit,paths=load_ratings();joblib.dump(raw,source,compress=3)
    cache=OUT/'prepared_role_panel.joblib'
    if reuse and cache.exists():
        raw,role,panel,variants,metadata_audit=joblib.load(cache)
        if 'learned_audit' not in role:
            learned,la,ls,lc=learn_profile_forecasts(raw,role['z'],role['profiles'])
            role['features']=pd.concat([role['features'],learned],axis=1)
            role.update(learned_audit=la,learned_forecasts=ls,forecast_coefficients=lc)
            for c in LEARNED:panel[c]=panel.row_id.map(learned.set_axis(raw.row_id)[c])
            variants={BASE:variants[BASE],STATIC_MODEL:variants[STATIC_MODEL],LINEAR:variants['Player + future role'],NEW:variants[BASE]+STATIC+LEARNED}
            joblib.dump((raw,role,panel,variants,metadata_audit),cache,compress=3)
    else:
        role=build_role_panel(raw);panel,variants,metadata_audit=make_panel(raw,role)
        joblib.dump((raw,role,panel,variants,metadata_audit),cache,compress=3)
    validation=validate_time(raw,role)
    role['profile_history'].to_csv(OUT/'role_profiles.csv',index=False);role['trend_audit'].to_csv(OUT/'trend_time_audit.csv',index=False)
    role['profile_forecasts'].to_csv(OUT/'role_forecast_accuracy.csv',index=False);metadata_audit.to_csv(OUT/'league_name_audit.csv',index=False)
    role['learned_audit'].to_csv(OUT/'learned_profile_time_audit.csv',index=False);role['learned_forecasts'].to_csv(OUT/'learned_profile_accuracy.csv',index=False)
    role['forecast_coefficients'].to_csv(OUT/'learned_forecast_coefficients.csv',index=False)
    forecasts,metrics,bundles,audit=fit_models(panel,variants,YEARS+[2026],reuse=reuse)
    assert audit.last_label.le(audit.year).all() and audit.last_origin.le(audit.year-3).all()
    assert forecasts.overall.lt(70).all() and not forecasts.duplicated(['edition','player_id']).any()
    if reuse:
        indexed=panel.set_index('row_id')
        for year,g in forecasts.groupby('edition'):
            sample=g.head(8);profiles=indexed.reindex(sample.row_id)
            for name in [BASE,STATIC_MODEL,LINEAR,NEW]:
                bundle=bundles[year][name]
                score=bundle['model'].predict(model_frame(profiles,bundle['columns']),prediction_type='RawFormulaVal')
                np.testing.assert_allclose(score,sample[name],atol=1e-6,rtol=1e-6)
    intervals=uncertainty(forecasts)
    if reuse and (OUT/'profile_shift_uncertainty.csv').exists():shifts=pd.read_csv(OUT/'profile_shift_uncertainty.csv')
    else:shifts=profile_shift_bootstrap(role);shifts.to_csv(OUT/'profile_shift_uncertainty.csv',index=False)
    cases,contributions,neighbors,importance=named_cases(panel,forecasts,role,bundles)
    primary=metrics.loc[metrics.k.eq(50)&metrics.target.eq('quality80')]
    totals=primary.groupby('method').found.sum().astype(int).to_dict()
    mature=forecasts.loc[forecasts.edition.isin(YEARS)]
    unique={m:mature.loc[np.concatenate([ordered(f,m).head(50).index.to_numpy() for _,f in mature.groupby('edition')])].loc[lambda x:x.event_quality80.eq(1),'player_id'].nunique() for m in METHODS}
    forecast_check=role['learned_forecasts'].loc[role['learned_forecasts'].origin.isin(YEARS)]
    future_gain=totals[NEW]-totals[STATIC_MODEL]
    rare=metrics.loc[metrics.k.eq(50)&metrics.target.eq('star85')].groupby('method').found.sum().astype(int).to_dict()
    strict=pd.DataFrame([measure(f,m) for y,f in forecasts.loc[forecasts.edition.isin(YEARS)&forecasts.peak_seen.lt(70)].groupby('edition') for m in METHODS])
    strict.to_csv(OUT/'never_previously70_performance.csv',index=False)
    summary={'hits_at50':totals,'star85_hits':rare,'never70_at50':strict.groupby('method').found.sum().astype(int).to_dict(),
        'confirmed80_events':int(primary.groupby('year').confirmed_total.first().sum()),
        'unique80_players':{k:int(v) for k,v in unique.items()},'years_future_beats_static':int((primary.pivot(index='year',columns='method',values='found')[NEW]>primary.pivot(index='year',columns='method',values='found')[STATIC_MODEL]).sum()),
        'extra_future_vs_static':int(future_gain),'profile_forecast_wins':int(forecast_check.improvement.gt(0).sum()),'profile_forecast_comparisons':len(forecast_check),
        'feature_counts':{k:len(v) for k,v in variants.items()},'geometry_variance':role['normalizer']['explained_variance'],
        'validation':{**validation,'dated_training_labels':True,'native_shap_additivity':True},
        'retrospective_scope':'Previously inspected ratings archive. These are dated replays, not independent prospective validation.',
        'packages':{m:importlib.metadata.version(m) for m in ['numpy','pandas','scikit-learn','lightgbm','catboost','plotly','matplotlib']}}
    (OUT/'findings.json').write_text(json.dumps(summary,indent=2))
    (OUT/'experiment_protocol.json').write_text(json.dumps({'primary_outcome':'Reaches 80+ within the next three editions from current <70; any age; no prior 85+',
        'primary_budget':50,'primary_feature_test':[NEW,STATIC_MODEL],'secondary_test':[NEW,BASE],
        'features':variants,'parameters':CB_PARAMS,'trend_window':4,'trend_shrinkage':.5,'historical_geometry':2015,
        'missing_followup':'Confirmed positive when observed; negative requires all three records; remaining outcomes unresolved.'},indent=2))
    ordered(forecasts.loc[forecasts.edition.eq(2026)],NEW).head(100).to_csv(OUT/'fc26_snapshot_watchlist.csv',index=False)
    return dict(raw=raw,role=role,panel=panel,variants=variants,forecasts=forecasts,metrics=metrics,bundles=bundles,
                audit=audit,intervals=intervals,shifts=shifts,cases=cases,contributions=contributions,neighbors=neighbors,importance=importance,summary=summary)

def explain_player(study,player_id,edition=2023,method=BASE):
    row=study['panel'].loc[lambda p:p.player_id.eq(player_id)&p.edition.eq(edition)]
    if len(row)!=1:raise ValueError('Choose one recorded player and edition.')
    bundle=study['bundles'][edition][method];x=model_frame(row,bundle['columns'])
    values=bundle['model'].get_feature_importance(Pool(x),type='ShapValues')[0]
    score=bundle['model'].predict(x,prediction_type='RawFormulaVal')[0]
    np.testing.assert_allclose(values.sum(),score,atol=1e-6)
    out=pd.DataFrame({'feature':bundle['columns'],'value':row.iloc[0][bundle['columns']].to_numpy(),'shap':values[:-1]})
    out.attrs.update(expected=float(values[-1]),score=float(score),player_id=int(player_id),edition=int(edition),method=method)
    return out.loc[out.shap.abs().sort_values(ascending=False).index]

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display,Image
from sklearn.cluster import KMeans
import base64,html,gzip

INK='#173441';TEAL='#008777';GOLD='#c48b29';BLUE='#427fa3';RED='#bc625e';GREY='#a7b6bd';PALE='#e3eaed'
COLORS={FIFA:GOLD,CAL:GREY,BASE:INK,STATIC_MODEL:BLUE,LINEAR:'#b3a4bf',NEW:TEAL}
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.labelcolor':INK,'text.color':INK,'xtick.color':INK,'ytick.color':INK,'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':.13,'figure.facecolor':'white','axes.facecolor':'white','savefig.facecolor':'white'})


def nice(c):
    aliases={'overall':'Current overall','potential':'FIFA potential','potential_gap':'Potential headroom','age':'Age',
        'team_role_rating_depth_gap':'Gap to strongest same-role teammate','team_pct_role_quality':'Skill rank in club',
        'core_growth_ahead_of_rating':'Core skills growing ahead of rating','role_headroom':'Position skills above overall',
        'role_fit_now':'Current elite-profile similarity','role_distance_now':'Current elite-profile distance',
        'learned_future_fit':'Forecast profile similarity','learned_future_advantage':'Forecast fit advantage',
        'learned_future_distance':'Forecast profile distance','learned_future_alignment':'Alignment with forecast change',
        'learned_change_size':'Forecast change magnitude','learned_forecast_training_r2':'Historical forecast fit',
        'learned_forecast_transitions':'Past role transitions','learned_forecast_available':'Forecast available'}
    aliases.update({'position_age_pct_overall':'Rating rank · exact age + position','team_strength_global_percentile':'Club strength percentile',
        'team_top11_rating':'Club top-11 rating','team_median_rating':'Club median rating','league_best_team_strength':'Strongest club in league',
        'league_mean_team_strength':'Typical league club strength','club_named_starter':'Listed as starter',
        'role_quality_peer_percentile':'Role-skill rank · position + age band','mental_quality_peer_percentile':'Mental-skill rank · position + age band','height_cm':'Height',
        'position_age_pct_role_quality':'Role-skill rank · exact age + position','age_vs_team_mean':'Age relative to teammates',
        'position_pct_overall':'Rating rank within position','skill_fk_accuracy':'Free-kick accuracy','overall_past_slope':'Past rating trend',
        'team_roster_log_n':'Club squad size','age_rating_interaction':'Age and current rating'})
    if c in aliases:return aliases[c]
    return c.replace('attacking_','').replace('movement_','').replace('mentality_','').replace('defending_','').replace('skill_','').replace('power_','').replace('_',' ').capitalize()

def finish(fig,name,note=None):
    if note:fig.text(.015,.015,note,ha='left',va='bottom',fontsize=9,color='#637c88')
    fig.tight_layout(rect=[0,.06 if note else .01,1,.94]);fig.savefig(OUT/name,dpi=170,bbox_inches='tight')
    display(Image(filename=str(OUT/name)));plt.close(fig)

def plot_similarity(study):
    raw=study['raw'];role=study['role'];all_players=study['forecasts'].loc[study['forecasts'].edition.eq(2023)].copy()
    f=all_players.loc[all_players.position_group.ne('GK')].copy()
    co=role['coordinates'];rng=np.random.default_rng(SEED);sample=f.sample(min(1500,len(f)),random_state=SEED)
    fig,ax=plt.subplots(figsize=(12,6.4));xy=co[sample.row_id.to_numpy()];ax.scatter(xy[:,0],xy[:,1],s=8,color=GREY,alpha=.25,rasterized=True)
    hits=f.loc[f.event_quality80.eq(1)];xy=co[hits.row_id.to_numpy()];ax.scatter(xy[:,0],xy[:,1],s=45,color=TEAL,label='Eventually reaches 80+',zorder=3)
    ml=ordered(all_players,BASE).head(50).loc[lambda x:x.position_group.ne('GK')];xy=co[ml.row_id.to_numpy()];ax.scatter(xy[:,0],xy[:,1],s=75,facecolors='none',edgecolors=INK,lw=1.2,label='Player-model top 50',zorder=4)
    fifa=ordered(all_players,FIFA).head(50).loc[lambda x:x.position_group.ne('GK')];xy=co[fifa.row_id.to_numpy()];ax.scatter(xy[:,0],xy[:,1],s=100,facecolors='none',edgecolors=GOLD,lw=1,label='FIFA top 50',zorder=3)
    offsets=[(18,28),(-150,32),(25,-35),(-140,-35)]
    named=study['cases'].loc[study['cases'].position_group.ne('GK')]
    for (_,r),offset in zip(named.iterrows(),offsets):
        point=co[int(r.row_id)];color=TEAL if r.event_quality80==1 else RED
        ax.scatter(*point,s=65,color=color,zorder=6)
        ax.annotate(f'{r.short_name}\n{r.overall:.0f} → {r.observed_peak3:.0f} · ML #{r.ml_rank:.0f}',point,xytext=offset,textcoords='offset points',
                    fontsize=9,fontweight='bold',color=color,bbox=dict(boxstyle='round,pad=.35',facecolor='white',edgecolor=PALE,alpha=.95),
                    arrowprops=dict(arrowstyle='-',color=color,lw=1),zorder=7)
    ev=role['normalizer']['explained_variance'];ax.set_xlabel(f'Principal component 1 · {100*ev[0]:.1f}% of profile variation');ax.set_ylabel(f'Principal component 2 · {100*ev[1]:.1f}%')
    ax.legend(loc='upper right',fontsize=9,frameon=True,facecolor='white');ax.set_title('Similar profiles, different futures · the 2023 scouting pool',fontsize=18,fontweight='bold',pad=18)
    finish(fig,'08_player_similarity.png','Outfield projection of the all-position shortlists. Coordinates fitted in 2015; future colours are revealed after prediction. Neighbors use all 28 dimensions.')

def plot_careers(study):
    cases=study['cases'];raw=study['raw'];fig,axs=plt.subplots(1,len(cases),figsize=(14,4.8),sharey=True)
    for ax,(_,r) in zip(np.atleast_1d(axs),cases.iterrows()):
        g=raw.loc[raw.player_id.eq(r.player_id)].sort_values('edition');color=TEAL if r.event_quality80==1 else RED
        ax.plot(g.edition,g.overall,'o-',color=color,lw=2);ax.axvline(2023,color=INK,ls='--');ax.axhline(80,color=GOLD,ls=':',lw=1.5)
        ax.axvspan(2023.05,2026.1,color=PALE,alpha=.5);ax.set_xticks([2021,2023,2026]);ax.set_xlim(min(2021,g.edition.min())-.4,2026.5)
        ax.set_title(f'{r.short_name}\nOVR {r.overall:.0f} · POT {r.potential:.0f}',fontweight='bold',fontsize=11)
        ax.text(.04,.04,f'ML #{r.ml_rank:.0f} | FIFA #{r.fifa_rank:.0f}\n{r.case}',transform=ax.transAxes,fontsize=9,color=color)
    np.atleast_1d(axs)[0].set_ylabel('Observed game rating');np.atleast_1d(axs)[0].set_ylim(45,90)
    fig.suptitle('The career reveal includes discoveries, a false alarm and a miss',fontsize=18,fontweight='bold')
    finish(fig,'09_player_careers.png','All examples use 2023 information. Cases are selected after evaluation to explain results; full rankings and denominators remain available.')

def plot_case_shap(study):
    cases=study['cases'].head(3);allc=study['contributions'];fig,axs=plt.subplots(1,len(cases),figsize=(14,4.6))
    for ax,(_,r) in zip(np.atleast_1d(axs),cases.iterrows()):
        c=allc.loc[allc.player_id.eq(r.player_id)].copy();c=c.loc[c.shap.abs().nlargest(6).index].sort_values('shap')
        ax.barh([nice(x)[:38] for x in c.feature],c.shap,color=np.where(c.shap.ge(0),TEAL,RED));ax.axvline(0,color=GREY)
        ax.set_title(r.short_name,fontweight='bold');ax.set_xlabel('SHAP contribution')
    fig.suptitle('Why these players moved up the scouting list',fontsize=18,fontweight='bold')
    finish(fig,'10_player_explanations.png','Six largest individual feature attributions, not the complete score. Ranking scores are not calibrated probabilities of becoming a star.')

READER_CSS='<style>\n.code_cell>.input,.jp-Cell-inputWrapper{display:none!important}.output_prompt,.input_prompt{display:none!important}\n.code_cell:has(>.input):not(:has(.output_area)){display:none!important}\n.future-note{font:13px/1.65 system-ui;color:#637c88}.future-finding{border-left:4px solid #008777;background:#ecf7f3;padding:16px 20px;margin:18px 0;color:#173441;font:16px/1.65 system-ui}\n.future-cards{display:grid;grid-template-columns:repeat(3,1fr);gap:13px;margin:19px 0}.future-cards>div{padding:18px;background:#f3f6f7;border-top:3px solid #008777;border-radius:8px}.future-cards b{display:block;font-size:30px;line-height:1.5}.future-cards span{font-size:12px;color:#637c88}\n.future-table{border-collapse:collapse;width:100%;font-size:13px;margin:14px 0}.future-table td,.future-table th{padding:10px;border-bottom:1px solid #dce5e9;text-align:left}.future-table th{background:#f1f5f6}.future-audit{overflow:auto;max-height:430px}\ndetails{padding:14px 18px;background:#f5f7f8;border:1px solid #dce5e9;margin:16px 0;border-radius:8px}summary{cursor:pointer;font-weight:600;color:#173441}\n@media(max-width:650px){.future-cards{grid-template-columns:1fr}}\n</style>'

from plotly.offline import get_plotlyjs

def room_payload(study,reuse=True):
    cache=OUT/'room_payload.json'
    if reuse and cache.exists():
        saved=json.loads(cache.read_text())
        if saved.get('schema_version')==3:return saved
    raw=study['raw'];role=study['role'];forecast=study['forecasts'].copy();coords=role['coordinates']
    forecast['nationality_name']=forecast.nationality_name.fillna('Unknown');forecast['club_name']=forecast.club_name.fillna('No recorded club')
    km=KMeans(n_clusters=6,n_init=10,random_state=42).fit(role['z'][raw.edition.eq(2015)&raw.position_group.ne('GK')])
    clusters=km.predict(role['z'])
    forecast['x']=coords[forecast.row_id.to_numpy(),0];forecast['y']=coords[forecast.row_id.to_numpy(),1]
    forecast['cluster']=clusters[forecast.row_id.to_numpy()]
    methods=METHODS;rankcols=[]
    for j,m in enumerate(methods):
        c='rank'+str(j);rankcols.append(c);forecast[c]=0
        for year,g in forecast.groupby('edition'):
            r=ordered(g,m);forecast.loc[r.index,c]=np.arange(1,len(r)+1)
    cols=['player_id','edition','short_name','age','overall','potential','position_group','nationality_name','club_name',
          'x','y','event_quality80','complete3','observed_peak3','delta3_observed','cluster','learned_future_advantage','peak_seen']+rankcols
    def clean(x):
        if isinstance(x,(np.bool_,bool)):return bool(x)
        if pd.isna(x):return None
        if isinstance(x,(np.integer,int)):return int(x)
        if isinstance(x,(np.floating,float)):return round(float(x),5)
        return str(x)
    players=[[clean(x) for x in row] for row in forecast[cols].itertuples(index=False,name=None)]
    ids=set(forecast.player_id);histories={str(int(i)):[[int(r.edition),int(r.overall),None if pd.isna(r.potential) else int(r.potential)] for r in g.itertuples()] for i,g in raw.loc[raw.player_id.isin(ids)].groupby('player_id')}
    explanations={};features=study['panel'].set_index('row_id')
    for year,g in forecast.groupby('edition'):
        chosen=set()
        for m in [BASE,NEW,FIFA]:chosen.update(ordered(g,m).head(120).row_id)
        chosen.update(g.loc[g.event_quality80.eq(1),'row_id'])
        sample=features.reindex(sorted(chosen))
        for name in [BASE,NEW]:
            bundle=study['bundles'][int(year)][name];mat=model_frame(sample,bundle['columns'])
            shap=bundle['model'].get_feature_importance(Pool(mat),type='ShapValues')
            scores=bundle['model'].predict(mat,prediction_type='RawFormulaVal')
            np.testing.assert_allclose(shap.sum(axis=1),scores,atol=1e-6)
            for i,(_,r) in enumerate(sample.iterrows()):
                a=shap[i,:-1];top=np.argsort(-np.abs(a))[:7]
                explanations[f'{int(year)}:{int(r.player_id)}:{methods.index(name)}']={
                    'expected':round(float(shap[i,-1]),5),'score':round(float(scores[i]),5),
                    'features':[[nice(bundle['columns'][j]),round(float(a[j]),5)] for j in top],
                    'other':round(float(a.sum()-a[top].sum()),5)}
    neighbors={str(int(i)):g[['neighbor','neighbor_overall','distance','dimensions']].to_dict('records') for i,g in study['neighbors'].groupby('player_id')}
    pca=role['normalizer']['pca'];skills=role['normalizer']['skills'];axes=[]
    for v in pca.components_:
        pos=[nice(skills[j]) for j in np.argsort(-v)[:3]];neg=[nice(skills[j]) for j in np.argsort(v)[:3]]
        axes.append({'positive':pos,'negative':neg})
    payload={'schema_version':3,'columns':cols,'players':players,'methods':methods,'years':YEARS+[2026],'base':methods.index(BASE),'future':methods.index(NEW),'fifa':methods.index(FIFA),
             'histories':histories,'explanations':explanations,'neighbors':neighbors,'variance':role['normalizer']['explained_variance'],
             'axis_loadings':axes,'cluster_names':['Profile '+str(i+1) for i in range(6)]}
    cache.write_text(json.dumps(payload,separators=(',',':'),ensure_ascii=False));return payload

def export_room(study,reuse=True):
    payload=room_payload(study,reuse)
    packed=base64.b64encode(gzip.compress(json.dumps(payload,separators=(',',':'),ensure_ascii=False).encode(),compresslevel=9)).decode()
    template=globals().get('ROOM_TEMPLATE')
    logic=globals().get('ROOM_LOGIC')
    if template is None:template=(Path(__file__).parent/'scout_room.html').read_text()
    if logic is None:logic=(Path(__file__).parent/'scout_room.js').read_text()
    page=template.replace('__PLOTLY__',get_plotlyjs()).replace('__PACKED__',packed).replace('__LOGIC__',logic)
    path=OUT/'Beyond_FIFA_Potential_Scout_Room.html';path.write_text(page);return path

import re

ABLATIONS=['FIFA inputs only','Without team / peer context','Without player history']

def feature_removal_intervals(forecasts,draws=500):
    ids,iv=np.unique(forecasts.player_id,return_inverse=True)
    rng=np.random.default_rng(904);arrays={};methods=[BASE]+ABLATIONS
    for year,g in forecasts.groupby('edition'):
        loc=forecasts.index.get_indexer(g.index)
        orders={m:np.array([g.index.get_loc(i) for i in ordered(g,m).index]) for m in methods}
        arrays[year]=(loc,orders,g.event_quality80.eq(1).to_numpy(float))
    samples={m:[] for m in ABLATIONS}
    for _ in range(draws):
        weights=rng.multinomial(len(ids),np.full(len(ids),1/len(ids)))[iv]
        delta={m:0. for m in ABLATIONS}
        for loc,orders,event in arrays.values():
            hits={m:weighted_hit(o,event,weights[loc],50) for m,o in orders.items()}
            for m in ABLATIONS:delta[m]+=hits[BASE]-hits[m]
        for m in ABLATIONS:samples[m].append(delta[m])
    rows=[]
    for m in ABLATIONS:
        effect=sum(measure(g,BASE)['found']-measure(g,m)['found'] for _,g in forecasts.groupby('edition'))
        rows.append({'model':BASE,'baseline':m,'extra_hits':effect,'low':np.quantile(samples[m],.025),'high':np.quantile(samples[m],.975),'draws':draws,'scope':'Conditional on fitted models and these historical cohorts; player IDs resampled jointly across editions.'})
    result=pd.DataFrame(rows);result.to_csv(OUT/'feature_removal_intervals.csv',index=False)
    return result

def scouting_feature_group(c):
    if c in ['potential','potential_gap']:
        return 'FIFA potential'
    if c.startswith(('team_','league_','club_','position_','rating_vs_team','age_vs_team')) or c.endswith('_peer_percentile'):
        return 'Team / peer context'
    if re.search(r'_(lag|change)[1-4]$',c) or '_past_' in c or c.startswith(('past_','recent_','rating_below_','rating_above_')) or c in ['history_length','rating_acceleration','core_growth_ahead_of_rating','technical_growth_ahead_of_rating','previous_peak_gap','first_observed_age','editions_since_first_seen']:
        return 'Player history'
    if c in ['age','height_cm','weight_kg','age_to_21','age_rating_interaction']:
        return 'Age / physique'
    return 'Current skills'

def add_feature_evidence(study,reuse=False):
    p=study['panel'];base=study['variants'][BASE]
    core=['overall','age','potential','potential_gap']+['role_'+r for r in ROLE_CORES]
    variants={ABLATIONS[0]:core,ABLATIONS[1]:[c for c in base if scouting_feature_group(c)!='Team / peer context'],ABLATIONS[2]:[c for c in base if scouting_feature_group(c)!='Player history']}
    feature_dict=pd.DataFrame([{'feature':c,'group':scouting_feature_group(c)} for c in base])
    feature_dict.to_csv(OUT/'feature_dictionary.csv',index=False)
    outputs=[];audits=[];fitted={};tick=time.monotonic()
    signature=hashlib.sha256(json.dumps({'variants':variants,'parameters':CB_PARAMS},sort_keys=True).encode()).hexdigest()
    for year in YEARS:
        cache=OUT/f'feature_ablation_{year}.joblib'
        if reuse and cache.exists():
            saved=joblib.load(cache)
            if saved.get('signature')==signature:
                outputs.append(saved['forecasts']);audits+=saved['audit'];fitted[year]=saved['models'];continue
        train=p.loc[p.fit_domain&p.outcome_end.le(year)&p.event_quality80.notna()]
        test=p.loc[p.eligible&p.edition.eq(year)&p.potential.notna()]
        out=study['forecasts'].loc[study['forecasts'].edition.eq(year)].copy()
        assert np.array_equal(out.row_id.to_numpy(),test.row_id.to_numpy())
        models={};audit=[]
        for name,cols in variants.items():
            if name==ABLATIONS[0] and (OUT/'same_learner_fifa_models.joblib').exists():
                bundle=joblib.load(OUT/'same_learner_fifa_models.joblib')[year]
                assert bundle['columns']==cols
                model=bundle['model']
            else:
                model=CatBoostClassifier(**CB_PARAMS)
                model.fit(model_frame(train,cols),train.event_quality80.astype(int),sample_weight=dated_weights(train))
            out[name]=model.predict(model_frame(test,cols),prediction_type='RawFormulaVal')
            models[name]={'model':model,'columns':cols}
            audit.append({'year':year,'method':name,'features':len(cols),'training_rows':len(train),'positives':int(train.event_quality80.sum()),'last_origin':int(train.edition.max()),'last_label':int(train.outcome_end.max())})
        joblib.dump({'signature':signature,'forecasts':out,'models':models,'audit':audit},cache,compress=3)
        outputs.append(out);audits+=audit;fitted[year]=models
        print(f'Feature controls {year}: {time.monotonic()-tick:.1f}s',flush=True)
    forecasts=pd.concat(outputs,ignore_index=True)
    methods=[FIFA]+ABLATIONS+[BASE]
    metrics=pd.DataFrame([measure(g,m,k) for _,g in forecasts.groupby('edition') for m in methods for k in [10,25,50,100,200,500]])
    metrics.to_csv(OUT/'feature_ablation_performance.csv',index=False)
    pd.DataFrame(audits).to_csv(OUT/'feature_ablation_time_audit.csv',index=False)
    assert all(a['last_label']<=a['year'] and a['last_origin']<=a['year']-3 for a in audits)
    forecasts.to_csv(OUT/'feature_ablation_rankings.csv',index=False)
    sample=p.loc[p.edition.eq(2023)&p.eligible].sample(min(1200,int((p.edition.eq(2023)&p.eligible).sum())),random_state=SEED)
    bundle=study['bundles'][2023][BASE]
    values=bundle['model'].get_feature_importance(Pool(model_frame(sample,base)),type='ShapValues')
    scores=bundle['model'].predict(model_frame(sample,base),prediction_type='RawFormulaVal')
    np.testing.assert_allclose(values.sum(axis=1),scores,atol=1e-6)
    np.savez_compressed(OUT/'feature_shap_2023.npz',row_id=sample.row_id.to_numpy(),columns=np.array(base),values=sample[base].to_numpy(float),shap=values[:,:-1],expected=values[:,-1],score=scores)
    importance=pd.DataFrame({'feature':base,'group':[scouting_feature_group(c) for c in base],'mean_abs_shap':np.abs(values[:,:-1]).mean(axis=0)})
    importance.to_csv(OUT/'player_feature_importance.csv',index=False)
    discoveries=[]
    for year,g in forecasts.groupby('edition'):
        mr=pd.Series(np.arange(1,len(g)+1),index=ordered(g,BASE).index)
        fr=pd.Series(np.arange(1,len(g)+1),index=ordered(g,FIFA).index)
        a=g.loc[g.event_quality80.eq(1),['player_id','short_name','edition','overall','potential','observed_peak3','position_group']].copy()
        a['ml_rank']=mr.reindex(a.index);a['fifa_rank']=fr.reindex(a.index)
        a['selection']=np.select([a.ml_rank.le(50)&a.fifa_rank.le(50),a.ml_rank.le(50),a.fifa_rank.le(50)],['Both','ML only','FIFA only'],default='Neither')
        discoveries.append(a)
    discoveries=pd.concat(discoveries);discoveries.to_csv(OUT/'confirmed_discovery_comparison.csv',index=False)
    case_values=study['panel'].set_index('row_id').reindex(study['cases'].row_id)
    case_values[['player_id','edition','short_name']+base].to_csv(OUT/'case_feature_values.csv',index=False)
    intervals=feature_removal_intervals(forecasts)
    result={'variants':variants,'metrics':metrics,'forecasts':forecasts,'audit':pd.DataFrame(audits),'importance':importance,'sample':sample,'shap':values[:,:-1],'expected':values[:,-1],'scores':scores,'columns':base,'discoveries':discoveries,'intervals':intervals}
    (OUT/'feature_evidence_summary.json').write_text(json.dumps({'top50_hits':metrics.loc[metrics.k.eq(50)].groupby('method').found.sum().astype(int).to_dict(),'features':{k:len(v) for k,v in variants.items()},'confirmed_selection_overlap':discoveries.selection.value_counts().to_dict(),'scope':'Fixed settings and dated labels; retrospective feature-removal comparison. Correlated features can substitute for each other.'},indent=2))
    study['feature_evidence']=result
    return study

from matplotlib.patches import Rectangle

def plot_decision_clock(study):
    fig,ax=plt.subplots(figsize=(12,3.6))
    items=[(2,2015,2020,INK,'Training profiles · outcomes finish by 2023'),(1,2015,2023,TEAL,'Player history and context known at the decision'),(0,2023,2026,GOLD,'Evaluation · the next three editions')]
    for y,start,end,color,label in items:
        ax.barh(y,end-start,left=start,height=.34,color=color)
        ax.text(2014.7,y,label,ha='right',va='center',fontsize=10)
    ax.axvline(2023,color=INK,ls='--');ax.text(2023,2.55,'2023 decision',ha='center',fontweight='bold')
    ax.set(xlim=(2014.7,2026.4),ylim=(-.6,2.8),yticks=[],xticks=np.arange(2015,2027),xlabel='Game edition');ax.grid(axis='y',visible=False)
    fig.suptitle('Every shortlist uses information available at the time',fontsize=18,fontweight='bold')
    finish(fig,'01_prediction_clock.png','The same expanding-window rule is applied to each forecast edition. Player IDs link records; names and IDs are excluded from the model.')

def main_metrics(study):
    return study['metrics'].loc[lambda q:q.k.eq(50)&q.target.eq('quality80')&q.method.isin([BASE,FIFA])].copy()

def plot_model_comparison(study):
    q=main_metrics(study);fig,axs=plt.subplots(1,2,figsize=(13,5));x=np.arange(4)
    for offset,name in [(-.18,FIFA),(.18,BASE)]:
        v=q.loc[q.method.eq(name)].set_index('year').reindex(YEARS).found
        bars=axs[0].bar(x+offset,v,width=.34,color=COLORS[name],label=name)
        for b,n in zip(bars,v):axs[0].text(b.get_x()+b.get_width()/2,n+.16,str(int(n)),ha='center',fontsize=11)
    axs[0].set(xticks=x,xticklabels=YEARS,ylabel='Confirmed future 80+ players in the top 50',ylim=(0,11));axs[0].legend(frameon=False,fontsize=9)
    axs[0].set_title('50 reviews in every edition')
    for name in [FIFA,BASE]:
        a=study['metrics'].loc[lambda m:m.method.eq(name)&m.target.eq('quality80')].groupby('k').found.sum()
        axs[1].plot(a.index,a.values,lw=2.5,marker='o',color=COLORS[name],label=name)
    axs[1].axvline(50,color=GREY,ls='--');axs[1].set(xlabel='Reviews per edition',ylabel='Confirmed 80+ selections across four lists')
    axs[1].set_title('The comparison across review budgets')
    fig.suptitle('The same scouting budget finds 22 future 80+ selections instead of 14',fontsize=17,fontweight='bold')
    finish(fig,'02_model_comparison.png','2020–2023. Every candidate starts below 70. Repeated players count once per annual shortlist; unresolved outcomes retain their slots.')

def plot_sampling_uncertainty(study):
    q=study['intervals'].loc[lambda x:x.model.eq(BASE)&x.baseline.eq(FIFA)]
    fig,ax=plt.subplots(figsize=(11,4.5))
    ax.errorbar(q.extra_hits,np.arange(len(q)),xerr=np.vstack([q.extra_hits-q.low,q.high-q.extra_hits]),fmt='o',color=TEAL,capsize=5,markersize=8)
    ax.axvline(0,color=INK,lw=1);ax.set_yticks(np.arange(len(q)),q.year.astype(str));ax.invert_yaxis();ax.set_xlabel('Additional confirmed 80+ selections · ML minus FIFA potential')
    fig.suptitle('The pooled gain is +8; the conditional 95% interval is +1 to +15',fontsize=17,fontweight='bold')
    finish(fig,'03_sampling_uncertainty.png','500 paired player-cluster bootstrap draws. Shared player IDs are resampled jointly; each full annual ranking is reselected at 50.')

def plot_feature_removal(study):
    e=study['feature_evidence'];q=e['metrics'].loc[lambda x:x.k.eq(50)].groupby('method').found.sum()
    methods=[FIFA,ABLATIONS[0],ABLATIONS[1],ABLATIONS[2],BASE]
    labels=['FIFA potential','Age, rating, potential + position','Without team / peer context','Without player history','Full player model']
    fig,axs=plt.subplots(1,2,figsize=(14,5.2),gridspec_kw={'width_ratios':[1.25,1]})
    bars=axs[0].barh(np.arange(5),[q[m] for m in methods],color=[GOLD,GREY,GREY,GREY,TEAL])
    axs[0].set_yticks(np.arange(5),labels,fontsize=10);axs[0].invert_yaxis();axs[0].set_xlim(0,max(q)*1.18)
    for b,m in zip(bars,methods):axs[0].text(q[m]+.3,b.get_y()+b.get_height()/2,str(int(q[m])),va='center',fontweight='bold')
    axs[0].set_xlabel('Confirmed 80+ selections · 200 reviews');axs[0].set_title('Retrain after removing a feature group')
    a=study['intervals'].loc[lambda x:x.model.eq(BASE)&x.baseline.eq(FIFA)&x.year.astype(str).eq('Pooled'),['baseline','extra_hits','low','high']]
    a=pd.concat([a,e['intervals'][['baseline','extra_hits','low','high']]],ignore_index=True)
    axs[1].errorbar(a.extra_hits,np.arange(len(a)),xerr=np.vstack([a.extra_hits-a.low,a.high-a.extra_hits]),fmt='o',color=INK,capsize=4,markersize=7)
    axs[1].axvline(0,color=GREY);axs[1].set_yticks(np.arange(4),['vs FIFA potential','vs FIFA inputs only','vs no context','vs no player history'],fontsize=10);axs[1].invert_yaxis()
    axs[1].set_xlabel('Full model: additional confirmed selections');axs[1].set_title('Paired conditional 95% intervals')
    fig.suptitle('Team and peer context make the largest difference in the removal check',fontsize=17,fontweight='bold')
    finish(fig,'04_feature_removal.png','All learned controls use the same target, dated training rows, weights and CatBoost settings. Removal losses overlap because features interact.')

def plot_feature_shap(study):
    e=study['feature_evidence'];imp=e['importance'].nlargest(12,'mean_abs_shap');cols=e['columns'];sample=e['sample'];rng=np.random.default_rng(42)
    fig,ax=plt.subplots(figsize=(12,6.5));cmap=plt.get_cmap('coolwarm')
    for i,c in enumerate(imp.feature):
        j=cols.index(c);v=sample[c];colour=v.rank(pct=True).to_numpy();sh=e['shap'][:,j];valid=v.notna().to_numpy()
        jitter=rng.normal(0,.11,len(sh));ax.scatter(sh[valid],i+jitter[valid],c=colour[valid],vmin=0,vmax=1,cmap=cmap,s=11,alpha=.5,edgecolors='none',rasterized=True)
        if (~valid).any():ax.scatter(sh[~valid],i+jitter[~valid],c=GREY,s=10,alpha=.4)
    ax.axvline(0,color=GREY,lw=1);ax.set_yticks(np.arange(len(imp)),[nice(c) for c in imp.feature]);ax.invert_yaxis()
    ax.set_xlabel('Contribution to the player-model score ← lower priority | higher priority →');ax.grid(axis='y',visible=False)
    sm=plt.cm.ScalarMappable(cmap=cmap,norm=plt.Normalize(0,1));cb=fig.colorbar(sm,ax=ax,pad=.025,fraction=.028);cb.set_ticks([0,1],labels=['Low','High']);cb.set_label('Feature value within the sample')
    fig.suptitle('Potential still matters. Context changes the ranking.',fontsize=18,fontweight='bold')
    finish(fig,'05_feature_shap.png','Native SHAP · 1,200 randomly sampled eligible 2023 profiles. Features ordered by mean absolute contribution. Grey points indicate missing values.')

def plot_feature_patterns(study):
    e=study['feature_evidence'];fig,axs=plt.subplots(1,2,figsize=(12,4.8))
    for ax,c in zip(axs,['team_top11_rating','position_age_pct_overall']):
        j=e['columns'].index(c);x=e['sample'][c].to_numpy();y=e['shap'][:,j];ok=np.isfinite(x);x=x[ok];y=y[ok]
        if c.endswith('pct_overall'):x=x*100
        ax.scatter(x,y,s=13,alpha=.2,color=BLUE,rasterized=True)
        t=pd.DataFrame({'value':x,'shap':y});t['bin']=pd.qcut(t.value,8,duplicates='drop')
        med=t.groupby('bin',observed=True)[['value','shap']].median();ax.plot(med.value,med.shap,color=TEAL,lw=2.3,marker='o',markersize=5)
        ax.axhline(0,color=GREY,lw=1);ax.set_xlabel('Club top-11 rating' if c=='team_top11_rating' else 'Rating rank among the same age + position · percentile')
        ax.set_ylabel('Contribution to the model score');ax.set_title('The club around the player' if c=='team_top11_rating' else 'A low overall rating in the right peer group')
    fig.suptitle('The extra signal is relative to the player’s surroundings',fontsize=17,fontweight='bold')
    finish(fig,'06_feature_patterns.png','Each point is one 2023 profile; lines connect median contributions in value bins. These show fitted-model associations, not effects of changing clubs or ranks.')

def plot_discovery_ranks(study):
    d=study['feature_evidence']['discoveries'];fig,ax=plt.subplots(figsize=(12,6.5));limit=max(d.ml_rank.max(),d.fifa_rank.max())*1.6
    ax.add_patch(Rectangle((50,.75),limit-50,49.25,color=TEAL,alpha=.06,zorder=0));ax.add_patch(Rectangle((.75,50),49.25,limit-50,color=GOLD,alpha=.06,zorder=0))
    for label,color,size in [('Neither',GREY,24),('Both',INK,60),('FIFA only',GOLD,75),('ML only',TEAL,75)]:
        g=d.loc[d.selection.eq(label)];ax.scatter(g.fifa_rank,g.ml_rank,s=size,color=color,alpha=.75,label=f'{label} · {len(g)}',zorder=3)
    ax.axhline(50,color=INK,ls='--',lw=1);ax.axvline(50,color=INK,ls='--',lw=1)
    ax.set_xscale('log');ax.set_yscale('log');ax.set_xlim(.75,limit);ax.set_ylim(limit,.75)
    ax.set_xlabel('FIFA potential rank · logarithmic scale');ax.set_ylabel('Player-model rank · logarithmic scale')
    selected=d.loc[d.edition.eq(2023)&d.short_name.isin(['M. Svilar','M. Kerkez','A. Güler'])].copy()
    extra=d.loc[d.selection.eq('ML only')].assign(gain=lambda g:g.fifa_rank/g.ml_rank).sort_values('gain',ascending=False)
    selected=pd.concat([selected,extra]).drop_duplicates('player_id').head(6)
    offsets=[(12,-18),(12,14),(-70,15),(10,-20),(10,18),(-90,-15)]
    for ((_,r),offset) in zip(selected.iterrows(),offsets):
        ax.annotate(f'{r.short_name} · {str(r.edition)[-2:]}',(r.fifa_rank,r.ml_rank),xytext=offset,textcoords='offset points',fontsize=9,color=INK,arrowprops={'arrowstyle':'-','color':GREY,'lw':.8},bbox={'boxstyle':'round,pad=.25','fc':'white','ec':'none','alpha':.92})
    ax.legend(loc='lower right',frameon=True,fontsize=9);fig.suptitle('The model finds 11 breakouts FIFA misses; FIFA finds 3 the model misses',fontsize=16,fontweight='bold')
    finish(fig,'07_discovery_ranks.png','All 101 confirmed 80+ player-edition events. Dashed lines mark top 50. Names illustrate the ranking differences; repeated players can appear in different editions.')

def comparison_table(study):
    q=main_metrics(study);rows=[]
    for m in [BASE,FIFA]:
        g=q.loc[q.method.eq(m)];hits=int(g.found.sum());selected=int(g.selected.sum());total=int(g.confirmed_total.sum())
        rows.append({'Ranking':m,'Confirmed 80+':hits,'Distinct players':study['summary']['unique80_players'][m],'Reviews':selected,'Confirmed precision':f'{hits/selected:.1%}','Confirmed recall':f'{hits/total:.1%}'})
    return pd.DataFrame(rows).to_html(index=False,border=0,classes='future-table')

def annual_table(study):
    q=main_metrics(study);rows=[]
    for year,g in q.groupby('year'):
        r={'Edition':int(year),'Eligible players':int(g.eligible.iloc[0]),'Confirmed 80+':int(g.confirmed_total.iloc[0])}
        for method,short in [(BASE,'ML'),(FIFA,'FIFA')]:
            a=g.loc[g.method.eq(method)].iloc[0];r[short+' found / 50']=int(a.found);r[short+' precision']=f'{a.precision_lower:.1%}';r[short+' recall']=f'{a.confirmed_recall:.1%}';r[short+' unresolved']=int(a.unresolved)
        rows.append(r)
    return '<details><summary>Annual precision, recall and outcome counts</summary><div class="future-audit">'+pd.DataFrame(rows).to_html(index=False,border=0,classes='future-table')+'</div></details>'

def section_text(key,study):
    s=study['summary'];e=study['feature_evidence'];hits=s['hits_at50'];counts=e['metrics'].loc[lambda x:x.k.eq(50)].groupby('method').found.sum()
    if key=='question':
        return '''<p>FIFA potential is a natural place to start a scouting shortlist. I wanted to test whether the same review budget could find more future quality when I added a player's skills, development and surroundings.</p>
        <p>I ranked players with <b>overall below 70, at any age</b>, and checked whether they reached <b>80+ in the next three editions</b>. Previously observed 85+ players were excluded. I compared 50 selections per edition across 2020–2023.</p>
        <p>The model uses FIFA potential as one input. The question is whether the additional information improves the order in which players deserve attention.</p>'''
    if key=='timing':
        return '''<p>I trained each model using only outcomes available at its decision date. For a 2023 shortlist, training profiles end in 2020, with their three-edition outcomes known by 2023. The shortlist is then evaluated using 2024–2026 ratings.</p>
        <p>A recorded 80+ rating confirms a positive outcome. A negative requires all three future records. Players with incomplete follow-up stay unresolved: disappearing from the dataset does not count as stagnation.</p>'''
    if key=='comparison':
        return f'''<div class="future-finding"><b>The player model finds {hits[BASE]} confirmed future 80+ selections; FIFA potential finds {hits[FIFA]}.</b> That is eight additional hits from the same 200 review slots, or {hits[BASE]/hits[FIFA]-1:.0%} more confirmed hits.</div>
        {comparison_table(study)}
        <p>The model matches or exceeds potential in each annual top-50 list: 4 versus 1, 5 versus 5, 4 versus 3, and 9 versus 5. The budget curve shows how the comparison changes as the shortlist grows.</p>
        <p class="future-note">Some players recur across editions. Confirmed precision uses all review slots; confirmed recall uses the 101 documented positive player-edition outcomes. Unresolved players remain in both rankings.</p>'''
    if key=='uncertainty':
        a=study['intervals'].loc[lambda x:x.model.eq(BASE)&x.baseline.eq(FIFA)&x.year.astype(str).eq('Pooled')].iloc[0]
        return f'''<p>I resampled whole player IDs across all four editions and rebuilt the top-50 lists in each draw. This preserves the connection between repeated observations of the same player.</p>
        <p>The pooled difference is <b>+8 confirmed hits</b>, with a conditional 95% interval of <b>{a.low:+.0f} to {a.high:+.0f}</b>. Individual seasons have wider intervals. This calculation holds the fitted models and historical cohorts fixed; it does not include model-selection uncertainty or variation across future seasons.</p>'''
    if key=='removal':
        ci=e['intervals'].set_index('baseline')
        return f'''<p>I retrained the same model after removing two groups of information. The target, training dates, weights and CatBoost settings stayed fixed.</p>
        <p>Without <b>team and peer context</b>, confirmed hits fall from {hits[BASE]} to <b>{int(counts[ABLATIONS[1]])}</b>. Without <b>player history</b>, they fall to <b>{int(counts[ABLATIONS[2]])}</b>. A model using only age, current rating, potential, headroom and position finds <b>{int(counts[ABLATIONS[0]])}</b>.</p>
        <p>The conditional interval for the context difference is <b>{ci.loc[ABLATIONS[1],'low']:+.0f} to {ci.loc[ABLATIONS[1],'high']:+.0f}</b> hits. The history difference is less precise: <b>{ci.loc[ABLATIONS[2],'low']:+.0f} to {ci.loc[ABLATIONS[2],'high']:+.0f}</b>.</p>
        <p>The largest drop comes from removing context: club strength, standing within the squad, and comparisons with players of the same position and age. The removal losses overlap because the remaining features are retrained together.</p>'''
    if key=='features':
        return '''<p><b>Potential remains the strongest individual feature.</b> The added information changes how that potential is interpreted. Among the leading inputs are club top-11 strength, listed squad status, rating rank for the same age and position, and position-specific skill rank.</p>
        <p>The chart shows how each feature contributes to the fitted ranking. Points to the right raise a player's score; colour shows whether the feature value is relatively low or high. Importance is measured on a random sample of 1,200 eligible 2023 players, rather than only successful discoveries.</p>
        <p>Club strength and peer standing help distinguish players with similar overall ratings. These are predictive associations: a contribution from club strength does not establish that joining a stronger club causes development.</p>'''
    if key=='ranks':
        return '''<p>The difference between the lists is visible at player level. <b>Eleven confirmed breakouts appear only in the ML top 50; three appear only in FIFA's.</b> Another eleven appear in both. The net gain is eight selections.</p>
        <p>The rank plot includes all 101 confirmed player-edition breakouts. Players near the top right were prioritised by ML and ranked much lower by FIFA potential. Players near the bottom left show the reverse.</p>'''
    if key=='similarity':
        ev=100*sum(s['geometry_variance'])
        return f'''<p>I also compared players by the shape of their attributes. The coordinates were fitted on 2015 outfield profiles and then held fixed. The two displayed dimensions retain <b>{ev:.1f}%</b> of profile variation.</p>
        <p>Similar profiles can still have different outcomes. The map combines similarity with the two shortlists and the later rating reveal. Nearest-player comparisons use all 28 normalised skill dimensions within the same position group; goalkeeper comparisons use five goalkeeping attributes.</p>'''
    if key=='explorer':
        return '''<p>The scouting replay lets me compare rankings within an edition, position or nationality. Filters rerank the full eligible pool. Clicking a player reveals the recorded career and an available individual explanation.</p>'''
    if key=='cases':
        t=study['cases'][['short_name','overall','potential','ml_rank','fifa_rank','observed_peak3','case']].copy();t.columns=['Player','Overall','Potential','ML rank','FIFA rank','Peak in window','Outcome']
        return '''<p>In 2023, <b>M. Kerkez</b> and <b>M. Svilar</b> both started at 69 and later reached 82. The model ranked them fifth and thirtieth; FIFA potential ranked them fifty-second and 376th.</p>
        <p>The same model ranked <b>Pablo Torre</b> first, but his observed peak in the window was 74. It also placed <b>A. Güler</b> sixty-second and missed him at top 50. These examples include discoveries, a false alarm and a miss.</p>'''+t.to_html(index=False,border=0,classes='future-table',float_format=lambda x:f'{x:.0f}')+'''
        <p>Individual SHAP values connect these rankings to specific inputs. They sum, with the expected score, to the fitted raw score. The displayed bars are the six largest contributions; the score is a ranking index rather than a calibrated probability.</p>'''
    if key=='findings':
        return f'''<div class="future-finding"><b>At a fixed review budget, adding player information improves the historical 80+ shortlist.</b> The gain is {hits[BASE]} versus {hits[FIFA]} confirmed selections. Team and peer context produce the largest loss when removed.</div>
        <p>The stricter pool of players never previously rated 70+ gives <b>{s['never70_at50'][BASE]} versus {s['never70_at50'][FIFA]}</b> confirmed selections. For the rarer 85+ outcome, the model and potential each find <b>{s['star85_hits'][BASE]}</b>; there are only ten documented 85+ player-edition events in these cohorts.</p>
        <p>I also tested forecast role-profile features. That extension finds {hits[NEW]} future 80+ selections at top 50, below the player model's {hits[BASE]}. The improvement over potential comes from the player model; the role forecast did not improve this shortlist.</p>
        <p>The scouting value is a better order for reviewing low-rated players. The next evaluation is a frozen pipeline on later-maturing cohorts, with match performance and playing time alongside game ratings.</p>'''
    raise KeyError(key)

def methods_html(study):
    dictionary=pd.read_csv(OUT/'feature_dictionary.csv');counts=dictionary.groupby('group').size().rename('Inputs').reset_index().rename(columns={'group':'Feature group'})
    return '''<details><summary>Data and methods</summary>
    <p>Men's annual FIFA15–FC26 snapshots. Exact player IDs link editions. Historical snapshots use the first available update; FC26 is the September 2025 snapshot. FC25 supplies future ratings but lacks potential.</p>
    <p>The player model has 431 inputs: current skills, age and physique, potential, past development, and contemporaneous team/peer context. Context uses edition-specific club and league IDs where available. Peer groups use exact age and primary position, or two-year age bands and a broader role. Percentiles are shrunk toward 0.5 for small groups. Names, player IDs, nationality and future outcomes are excluded from the predictors.</p>'''+counts.to_html(index=False,border=0,classes='future-table')+'''
    <p>CatBoost: 450 iterations, depth 5, learning rate 0.045, L2 12, border count 64, seed 42. Training includes sub-80 origins with no previously observed 85+ rating; labels must finish by the forecast edition. Sub-70 origins receive extra weight, repeated player histories are balanced, and rare large positive jumps receive bounded extra weight. All feature-removal models use the same settings and weights. Past club and peer changes remain in the context block; player history contains the player's own rating/skill changes, trends and observed career coverage.</p>
    <p>FIFA potential ranks descending; ties use higher current overall, younger age, then player ID. ML ties use player ID. Potential describes a longer horizon than the three-edition target; the comparison is at the same review budget.</p>
    <p>Precision is confirmed positives divided by all selections. The corresponding upper bound treats unresolved selections as positive. Recall uses documented positive outcomes. Source dropout can leave outcomes unknown. This is a retrospective replay on a previously explored archive, rather than untouched prospective validation. Bootstrap intervals condition on the fitted models and the four historical cohorts.</p>
    <p>The similarity transform, scaling and PCA are fitted in 2015. The outfield map excludes goalkeepers; scouting rankings include them. The interactive FC26 watchlist is based on the September 2025 source snapshot and has no mature three-edition outcomes.</p>
    <p>Sources: <a href="https://www.kaggle.com/datasets/stefanoleone992/ea-sports-fc-24-complete-player-dataset">Stefano Leone · FIFA15–FC24 · CC0</a>; <a href="https://www.kaggle.com/datasets/nyagami/ea-sports-fc-25-database-ratings-and-stats">Davis Nyagami · FC25 · Apache 2.0</a>; <a href="https://www.kaggle.com/datasets/rovnez/fc-26-fifa-26-player-data">rovnez · FC26 · CC BY 4.0</a>. Earlier research on skill evolution: <a href="https://dtai.cs.kuleuven.be/events/MLSA19/papers/verstraete_MLSA19.pdf">Verstraete et al., 2019</a>. Individual explanations use <a href="https://catboost.ai/docs/en/concepts/shap-values">native CatBoost SHAP</a>.</p>
    </details>'''

STORY=[('question','1. A shortlist before the breakthrough',[]),('timing','2. The prediction timeline',['01_prediction_clock.png']),('comparison','3. More discoveries from the same budget',['02_model_comparison.png']),('uncertainty','4. Uncertainty around the gain',['03_sampling_uncertainty.png']),('removal','5. The information behind the improvement',['04_feature_removal.png']),('features','6. The signals learned by the model',['05_feature_shap.png','06_feature_patterns.png']),('ranks','7. The players each approach finds',['07_discovery_ranks.png']),('similarity','8. Similar profiles, different futures',['08_player_similarity.png']),('explorer','9. An interactive scouting replay',[]),('cases','10. Discoveries, false alarms and misses',['09_player_careers.png','10_player_explanations.png']),('findings','11. What I found',[])]

def reading_room(room):
    return '<div class="future-room"><iframe title="Interactive hidden-gem scouting replay" sandbox="allow-scripts" style="width:100%;height:1650px;border:0" srcdoc="'+html.escape(room.read_text(),quote=True)+'"></iframe></div>'

def export_scouting_article(study,room):
    parts=[]
    for key,title,figures in STORY:
        part='<section><h2>'+title+'</h2>'+section_text(key,study)
        for name in figures:part+='<figure><img alt="'+html.escape(title)+'" src="data:image/png;base64,'+base64.b64encode((OUT/name).read_bytes()).decode()+'"></figure>'
        if key=='comparison':part+=annual_table(study)
        if key=='explorer':part+=reading_room(room)
        if key=='findings':part+=methods_html(study)
        parts.append(part+'</section>')
    css='''body{margin:0;background:#f2f5f6;color:#173441;font:16px/1.75 system-ui,-apple-system,Arial,sans-serif}header{background:#173441;color:white;padding:43px max(24px,calc((100% - 1140px)/2)) 36px}header small{letter-spacing:2px;text-transform:uppercase;color:#9ed8c7}header h1{font-size:37px;line-height:1.2;max-width:1000px}header p{color:#d5e4e8;max-width:1050px}main{max-width:1140px;margin:24px auto;background:#fff;padding:28px 40px;border:1px solid #dce5e9;border-radius:10px}h2{font-size:25px;line-height:1.4;margin:36px 0 17px}section{padding-bottom:15px}figure{margin:23px 0}figure img{width:100%;height:auto}a{color:#008777}@media(max-width:650px){main{padding:16px}header{padding:28px 18px}header h1{font-size:28px}}'''
    page='<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Beyond FIFA Potential: Finding Hidden Gems</title><style>'+css+'</style>'+READER_CSS+'</head><body><header><small>Football scouting · FIFA15–FC26</small><h1>Beyond FIFA potential: finding hidden gems before the breakthrough</h1><p>I compared a player model with FIFA potential to find future 80+ players while they were still rated below 70.</p></header><main>'+''.join(parts)+'</main></body></html>'
    path=OUT/'Beyond_FIFA_Potential_Article.html';path.write_text(page);return path

from datetime import date,datetime
from zoneinfo import ZoneInfo

RUN_DATE=os.environ.get('FIFA_ANALYSIS_DATE',datetime.now(ZoneInfo('Europe/Athens')).date().isoformat())
BUILTIN_SNAPSHOT_DATE='2025-09-21'

def snapshot_checks(frame):
    required=list(dict.fromkeys(['player_id','short_name','overall','potential','age','height_cm','weight_kg','player_positions','preferred_foot','club_name','league_name','club_position','weak_foot','skill_moves','pace','shooting','passing','dribbling','defending','physic']+SKILLS))
    missing=sorted(set(required)-set(frame.columns))
    if missing:raise ValueError('The current snapshot is missing required source fields: '+', '.join(missing))
    if frame.player_id.duplicated().any():raise ValueError('The current snapshot must contain one record per exact player ID.')
    for c in ['overall','potential','age','height_cm']:
        if pd.to_numeric(frame[c],errors='coerce').notna().mean()<.95:raise ValueError('Insufficient current coverage for '+c+'. No historical or invented potential is substituted.')
    for c in ['short_name','club_name','league_name','club_position']:
        if frame[c].notna().mean()<.9:raise ValueError('Insufficient current coverage for '+c+'.')
    if not pd.to_numeric(frame.overall,errors='coerce').between(1,99).all():raise ValueError('Overall ratings must be between 1 and 99.')
    return True

def load_current_snapshot(study):
    configured=os.environ.get('FIFA_CURRENT_SNAPSHOT_CSV','').strip()
    automatic=list(INPUT_ROOT.rglob('current_scouting_snapshot.csv')) if INPUT_ROOT.exists() else []
    if configured or automatic:
        path=Path(configured) if configured else automatic[0]
        if not configured and len(automatic)!=1:raise ValueError('Attach only one current_scouting_snapshot.csv.')
        current=pd.read_csv(path,low_memory=False);snapshot_checks(current)
        if 'edition' not in current:
            if 'fifa_version' not in current:raise ValueError('The current snapshot needs edition or fifa_version.')
            current['edition']=pd.to_numeric(current.fifa_version).astype(int)+2000
        if current.edition.nunique()!=1:raise ValueError('The current snapshot must contain a single game edition.')
        asof=os.environ.get('FIFA_CURRENT_SNAPSHOT_DATE','').strip()
        if not asof and 'snapshot_date' in current and current.snapshot_date.nunique()==1:asof=str(current.snapshot_date.iloc[0])
        if not asof:raise ValueError('Supply an ISO snapshot_date or FIFA_CURRENT_SNAPSHOT_DATE. The run date is not substituted for the data date.')
        asof=date.fromisoformat(asof).isoformat()
        if asof>RUN_DATE:raise ValueError('The snapshot date is later than the analysis run date.')
        current['source']=os.environ.get('FIFA_CURRENT_SOURCE_URL',path.name)
        meta={'snapshot_date':asof,'edition':int(current.edition.iloc[0]),'source':str(current.source.iloc[0]),'custom_snapshot':True}
        raw=pd.concat([study['raw'].loc[study['raw'].edition.lt(meta['edition'])],current],ignore_index=True)
        for c in study['raw'].columns:
            if c not in raw:raw[c]=np.nan
        for c in NUMERIC:raw[c]=pd.to_numeric(raw[c],errors='coerce')
        raw=raw.sort_values(['player_id','edition']).reset_index(drop=True);raw['row_id']=np.arange(len(raw))
        raw['primary_position']=raw.player_positions.fillna('Unknown').str.split(',').str[0].str.strip()
        mapping={'GK':'GK','CB':'CB','LB':'FB','RB':'FB','LWB':'FB','RWB':'FB','CDM':'MF','CM':'MF','CAM':'MF','LM':'Wing','RM':'Wing','LW':'Wing','RW':'Wing','ST':'FW','CF':'FW'}
        raw['position_group']=raw.primary_position.map(mapping).fillna('Other')
        clean,_,_=canonical_ids(raw)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore',pd.errors.PerformanceWarning)
            panel,_,_=strong_history_panel(clean)
        lookup=raw.set_index('row_id')
        for c in ['short_name','long_name','club_name','league_name','nationality_name']:
            panel[c]=panel.row_id.map(lookup[c])
    else:
        panel=study['panel'];meta={'snapshot_date':BUILTIN_SNAPSHOT_DATE,'edition':2026,'source':'rovnez / FC26_20250921.csv','custom_snapshot':False}
    trained=[year for year in study['bundles'] if int(year)<=meta['edition'] and BASE in study['bundles'][year]]
    meta['model_edition']=max(trained)
    meta['snapshot_age_days']=(date.fromisoformat(RUN_DATE)-date.fromisoformat(meta['snapshot_date'])).days
    return panel,meta

def readable_value(feature,value):
    if pd.isna(value):return 'not recorded'
    value=float(value)
    if 'percentile' in feature or '_pct_' in feature:return f'{100*value:.1f} percentile'
    if feature=='height_cm':return f'{value:.0f} cm'
    if feature=='weight_kg':return f'{value:.0f} kg'
    if feature=='age':return f'{value:.0f} years'
    if feature in ['club_named_starter','club_substitute','club_reserve']:return 'yes' if value>.5 else 'no'
    if abs(value-round(value))<1e-6:return f'{value:.0f}'
    return f'{value:.2f}'

def build_final_watchlist(study,top_n=20):
    panel,meta=load_current_snapshot(study);edition=meta['edition'];bundle=study['bundles'][meta['model_edition']][BASE]
    pool=panel.loc[panel.eligible&panel.edition.eq(edition)&panel.potential.notna()].copy()
    pool[BASE]=bundle['model'].predict(model_frame(pool,bundle['columns']),prediction_type='RawFormulaVal')
    pool[FIFA]=pool.potential
    if not meta['custom_snapshot']:
        saved=study['forecasts'].loc[study['forecasts'].edition.eq(edition)].set_index('player_id')
        np.testing.assert_allclose(pool[BASE],saved.reindex(pool.player_id)[BASE],atol=1e-6,rtol=1e-6)
    ranked=ordered(pool,BASE).copy();ranked['ml_rank']=np.arange(1,len(ranked)+1)
    fifa=ordered(pool,FIFA);fr=pd.Series(np.arange(1,len(fifa)+1),index=fifa.player_id)
    ranked['fifa_rank']=ranked.player_id.map(fr);top=ranked.head(top_n).copy()
    shap=bundle['model'].get_feature_importance(Pool(model_frame(top,bundle['columns'])),type='ShapValues')
    np.testing.assert_allclose(shap.sum(axis=1),top[BASE],atol=1e-6)
    rows=[];reasons=[];potcols=[j for j,c in enumerate(bundle['columns']) if c in ['potential','potential_gap']]
    for i,(_,player) in enumerate(top.iterrows()):
        values=shap[i,:-1];positive=np.flatnonzero(values>0);negative=np.flatnonzero(values<0)
        supports=positive[np.argsort(-values[positive])[:2]];brakes=negative[np.argsort(values[negative])[:1]]
        text=lambda j:f'{nice(bundle["columns"][j])}: {readable_value(bundle["columns"][j],player[bundle["columns"][j]])} ({values[j]:+.2f})'
        reasons.append({'player_id':int(player.player_id),'support':'; '.join(text(j) for j in supports),'brake':'; '.join(text(j) for j in brakes)})
        for j,c in enumerate(bundle['columns']):rows.append({'player_id':int(player.player_id),'player':player.short_name,'ml_rank':int(player.ml_rank),'feature':c,'feature_label':nice(c),'value':None if pd.isna(player[c]) else float(player[c]),'shap':float(values[j]),'group':'FIFA potential + headroom' if j in potcols else 'Other player / context inputs'})
    top=top.merge(pd.DataFrame(reasons),on='player_id',how='left',validate='one_to_one')
    top['snapshot_date']=meta['snapshot_date'];top['snapshot_edition']=edition
    top['raw_score']=top[BASE];top['expected_score']=shap[:,-1]
    top['potential_contribution']=shap[:,potcols].sum(axis=1);top['other_contribution']=shap[:,:-1].sum(axis=1)-top.potential_contribution
    top['past_editions_available']=top.overall_past_observations.sub(1).fillna(0).astype(int)
    exported=['ml_rank','player_id','short_name','age','position_group','club_name','overall','potential','fifa_rank','raw_score','expected_score','potential_contribution','other_contribution','past_editions_available','support','brake','snapshot_date','snapshot_edition']
    top[exported].to_csv(OUT/'latest_model_shortlist.csv',index=False);pd.DataFrame(rows).to_csv(OUT/'latest_shortlist_shap.csv',index=False)
    audit={'snapshot':meta,'eligible_players':len(pool),'top_n':len(top),'future_target_editions':[edition+1,edition+2,edition+3],'model':BASE,'features':len(bundle['columns']),'hyperparameter_search_performed':False,'probability_calibration_performed':False,'shap_additivity_checked':True,'outcomes':'Unconfirmed. No future ratings are used in the current shortlist.','parameters':bundle['model'].get_params()}
    (OUT/'latest_shortlist_audit.json').write_text(json.dumps(audit,indent=2))
    study['watchlist']={'top':top,'shap':pd.DataFrame(rows),'meta':meta,'audit':audit}
    return study

def current_shortlist_html(study):
    w=study['watchlist'];top=w['top'];meta=w['meta'];edition=meta['edition']
    dated=meta['snapshot_age_days']>90
    status='Dated snapshot · not a live current-season shortlist' if dated else 'Latest attached snapshot · prospective outcomes unconfirmed'
    header=f'<div class="future-finding"><b>{html.escape(status)}</b><br>Data as of {meta["snapshot_date"]} · FC{edition%100:02d} · {w["audit"]["eligible_players"]:,} eligible players · model: {BASE}.<br>Target: 80+ in editions {edition+1}–{edition+3}; all listed outcomes remain unconfirmed.</div>'
    rows=[]
    for _,r in top.iterrows():
        rows.append('<tr><td>'+str(int(r.ml_rank))+'</td><td><b>'+html.escape(r.short_name)+'</b><br><small>'+html.escape(str(r.club_name))+'</small></td><td>'+f'{r.age:.0f} · {r.position_group}'+'</td><td>'+f'{r.overall:.0f} / {r.potential:.0f}'+'</td><td>'+str(int(r.fifa_rank))+'</td><td>'+html.escape(r.support)+'</td><td>'+html.escape(r.brake)+'</td></tr>')
    table='<div style="overflow:auto"><table class="future-table latest-list"><thead><tr><th>ML rank</th><th>Player / club</th><th>Age / role</th><th>OVR / POT</th><th>FIFA rank</th><th>Main supporting signals · SHAP</th><th>Main opposing signal · SHAP</th></tr></thead><tbody>'+''.join(rows)+'</tbody></table></div>'
    details=[]
    for _,r in top.iterrows():
        c=w['shap'].loc[w['shap'].player_id.eq(r.player_id)].copy();c=c.loc[c.shap.abs().nlargest(8).index]
        t=pd.DataFrame({'Feature':c.feature_label,'Value':[readable_value(f,v) for f,v in zip(c.feature,c.value)],'Contribution':c.shap.map(lambda x:f'{x:+.3f}')})
        details.append(f'<details><summary>#{int(r.ml_rank)} · {html.escape(r.short_name)} — score detail</summary><p>FIFA potential and headroom contribute {r.potential_contribution:+.3f}; the other inputs contribute {r.other_contribution:+.3f}. Expected score {r.expected_score:+.3f} + all contributions = {r.raw_score:+.3f}. Earlier snapshots available within the previous four editions: {int(r.past_editions_available)}.</p>'+t.to_html(index=False,border=0,classes='future-table')+'<p class="future-note">Eight largest feature contributions shown. The complete decomposition is exported. Contributions describe the fitted model, not development causes or star probabilities.</p></details>')
    return header+table+''.join(details)

def plot_current_priorities(study):
    top=study['watchlist']['top'].head(12);fig,ax=plt.subplots(figsize=(12,6.2));y=np.arange(len(top))
    for i,(_,r) in enumerate(top.iterrows()):ax.plot([r.ml_rank,r.fifa_rank],[i,i],color=PALE,lw=3,zorder=1)
    ax.scatter(top.ml_rank,y,color=TEAL,s=55,label='Player model',zorder=3);ax.scatter(top.fifa_rank,y,color=GOLD,s=55,label='FIFA potential',zorder=3)
    ax.set_xscale('log');ax.set_yticks(y,top.short_name);ax.invert_yaxis();ax.set_xlabel('Rank in the same eligible pool · lower is reviewed earlier · logarithmic scale')
    ax.legend(frameon=False,loc='lower right');fig.suptitle('Where the model looks first in the latest complete snapshot',fontsize=17,fontweight='bold')
    finish(fig,'11_latest_shortlist_ranks.png',f'Data as of {study["watchlist"]["meta"]["snapshot_date"]}. Current overall below 70; all ages. These are unconfirmed prospective selections, not observed discoveries.')

def plot_current_player_reasons(study):
    w=study['watchlist'];top=w['top'].head(3);fig,axs=plt.subplots(1,len(top),figsize=(14,5))
    for ax,(_,r) in zip(np.atleast_1d(axs),top.iterrows()):
        c=w['shap'].loc[w['shap'].player_id.eq(r.player_id)].copy();c=c.loc[c.shap.abs().nlargest(6).index].sort_values('shap')
        ax.barh([x[:38] for x in c.feature_label],c.shap,color=np.where(c.shap.ge(0),TEAL,RED));ax.axvline(0,color=GREY)
        ax.set_title(f'#{int(r.ml_rank)} {r.short_name}\nOVR {r.overall:.0f} · POT {r.potential:.0f}',fontweight='bold',fontsize=11);ax.set_xlabel('SHAP contribution',fontsize=11)
    fig.suptitle('Why these profiles receive early scouting attention',fontsize=18,fontweight='bold')
    finish(fig,'12_latest_player_reasons.png','Six largest native SHAP contributions per player. The model includes FIFA potential; the other inputs change how that potential is prioritised.')

def selection_summary_html(study):
    metrics=study['metrics'].loc[lambda q:q.k.eq(50)&q.target.eq('quality80')].groupby('method').found.sum()
    control=int(study['feature_evidence']['metrics'].loc[lambda q:q.k.eq(50)&q.method.eq('FIFA inputs only')].found.sum())
    rows=[{'Candidate':FIFA,'Inputs':'Potential ranking','Confirmed 80+ selections':int(metrics[FIFA])},{'Candidate':'FIFA inputs only · CatBoost','Inputs':'11 · age / rating / potential / role','Confirmed 80+ selections':control}]
    for name in [BASE,STATIC_MODEL,LINEAR,NEW]:rows.append({'Candidate':name,'Inputs':str(len(study['variants'][name])),'Confirmed 80+ selections':int(metrics[name])})
    table=pd.DataFrame(rows).to_html(index=False,border=0,classes='future-table')
    stages=pd.DataFrame([['Validation','Four chronological expanding-window replays; every training outcome ends by the decision edition.'],['Selection criterion','Confirmed 80+ selections at top 50, with precision, recall, unique players, misses and conditional intervals alongside it.'],['Hyperparameters','One fixed CatBoost configuration for the player-feature comparisons. No automated hyperparameter search or early-stopping selection on the evaluation cohorts.'],['Fixed settings','450 iterations; depth 5; learning rate 0.045; L2 12; border count 64; seed 42.'],['Training weights','Sub-70 emphasis, balancing repeated player histories and bounded extra weight for rare large improvements.'],['Calibration','Not performed. The outputs are raw ranking scores, rather than calibrated probabilities.'],['Latest-snapshot refit','The selected specification is fitted to all eligible mature labels available by the forecast edition. The bundled FC26 model uses 66,811 training profiles from origins 2015–2023.'] ],columns=['Step','What I used'])
    return table+stages.to_html(index=False,border=0,classes='future-table')

def case_comparison_html(study):
    frame=study['cases'][['short_name','overall','potential','ml_rank','fifa_rank','observed_peak3','case']].copy()
    frame.columns=['Player','Overall','Potential','ML rank','FIFA rank','Peak in window','Outcome']
    return frame.to_html(index=False,border=0,classes='future-table',float_format=lambda x:f'{x:.0f}')

def newer_source_audit_html():
    rows=[['Current model-compatible archive','2025-09-21','Recorded','Used for the dated shortlist'],['EAFC26 Player Database · Flynn','2026-04-02','No potential field','Not substituted'],['EA FC 27 · EA API community export','2026-09-12','No potential; height / weight empty','Not substituted'],['FC27 · FIFPlay community export','2026-10-11','Potential empty in all 10,762 rows; duplicate IDs and mostly generic league labels','Not substituted']]
    frame=pd.DataFrame(rows,columns=['Source checked','Snapshot / file date','Compatibility','Use'])
    return '<details><summary>Source freshness and compatibility</summary>'+frame.to_html(index=False,border=0,classes='future-table')+'<p>Checked 2026-11-16. A file dated later is not enough: the model needs current potential, identity, age, physical data, skills and team context. Potential is not silently carried over from an older edition.</p></details>'

In [ ]:
from IPython.display import display,HTML,Image
from contextlib import redirect_stdout,redirect_stderr
import io
display(HTML(READER_CSS))
execution_log=io.StringIO()
reuse=os.environ.get('FUTURE_REUSE','0')=='1'
with redirect_stdout(execution_log),redirect_stderr(execution_log):
    study=execute_future_study(reuse=reuse)
    study=add_feature_evidence(study,reuse=reuse)
    study=build_final_watchlist(study,top_n=20)
    room=export_room(study,reuse=reuse)
log_bytes=(OUT/'execution_log.txt').write_text(execution_log.getvalue())


## 1. A shortlist before the breakthrough

FIFA potential is a natural place to start a scouting shortlist. I wanted to test whether the same review budget could find more future quality when I added a player's skills, development and surroundings.

I ranked players with **overall below 70, at any age**, and checked whether they reached **80+ in the next three editions**. Previously observed 85+ players were excluded. I compared 50 selections per edition across 2020–2023.

**The ML model uses FIFA potential and potential headroom as predictors.** I compare this combination with ranking by potential alone. The extra information comes from player skills, development history, club strength and relative standing among peers.

## 2. The prediction timeline

I trained each model using only outcomes available at its decision date. For a 2023 shortlist, training profiles end in 2020, with their three-edition outcomes known by 2023. The shortlist is then evaluated using 2024–2026 ratings.

A recorded 80+ rating confirms a positive outcome. A negative requires all three future records. Players with incomplete follow-up stay unresolved: disappearing from the dataset does not count as stagnation.

In [ ]:
plot_decision_clock(study)

## 3. More discoveries from the same budget

**The player model finds 22 confirmed future 80+ selections; FIFA potential finds 14.** That is eight additional hits from the same 200 review slots, or 57% more confirmed hits.

The model matches or exceeds potential in each annual top-50 list: 4 versus 1, 5 versus 5, 4 versus 3, and 9 versus 5. The budget curve shows how the comparison changes as the shortlist grows.

Some players recur across editions. Confirmed precision uses all review slots; confirmed recall uses the 101 documented positive player-edition outcomes. Unresolved players remain in both rankings.

In [ ]:
display(HTML(comparison_table(study)))
plot_model_comparison(study)
display(HTML(annual_table(study)))

## 4. Uncertainty around the gain

I resampled whole player IDs across all four editions and rebuilt the top-50 lists in each draw. This preserves the connection between repeated observations of the same player.

The pooled difference is **+8 confirmed hits**, with a conditional 95% interval of **+1 to +15**. Individual seasons have wider intervals. This calculation holds the fitted models and historical cohorts fixed; it does not include model-selection uncertainty or variation across future seasons.

In [ ]:
plot_sampling_uncertainty(study)

## 5. The information behind the improvement

I retrained the same model after removing two groups of information. The target, training dates, weights and CatBoost settings stayed fixed.

Without **team and peer context**, confirmed hits fall from 22 to **11**. Without **player history**, they fall to **17**. A model using only age, current rating, potential, headroom and position finds **7**.

The conditional interval for the context difference is **+3 to +19** hits. The history difference is less precise: **-1 to +11**.

The largest drop comes from removing context: club strength, standing within the squad, and comparisons with players of the same position and age. The removal losses overlap because the remaining features are retrained together.

In [ ]:
plot_feature_removal(study)

## 6. The signals learned by the model

**Potential remains the strongest individual feature.** The added information changes how that potential is interpreted. Among the leading inputs are club top-11 strength, listed squad status, rating rank for the same age and position, and position-specific skill rank.

The chart shows how each feature contributes to the fitted ranking. Points to the right raise a player's score; colour shows whether the feature value is relatively low or high. Importance is measured on a random sample of 1,200 eligible 2023 players, rather than only successful discoveries.

Club strength and peer standing help distinguish players with similar overall ratings. These are predictive associations: a contribution from club strength does not establish that joining a stronger club causes development.

In [ ]:
plot_feature_shap(study)
plot_feature_patterns(study)

## 7. The players each approach finds

The difference between the lists is visible at player level. **Eleven confirmed breakouts appear only in the ML top 50; three appear only in FIFA's.** Another eleven appear in both. The net gain is eight selections.

The rank plot includes all 101 confirmed player-edition breakouts. Players near the top right were prioritised by ML and ranked much lower by FIFA potential. Players near the bottom left show the reverse.

In [ ]:
plot_discovery_ranks(study)

## 8. Similar profiles, different futures

I also compared players by the shape of their attributes. The coordinates were fitted on 2015 outfield profiles and then held fixed. The two displayed dimensions retain **34.1%** of profile variation.

Similar profiles can still have different outcomes. The map combines similarity with the two shortlists and the later rating reveal. Nearest-player comparisons use all 28 normalised skill dimensions within the same position group; goalkeeper comparisons use five goalkeeping attributes.

In [ ]:
plot_similarity(study)

## 9. An interactive scouting replay

The scouting replay lets me compare rankings within an edition, position or nationality. Filters rerank the full eligible pool. Clicking a player reveals the recorded career and an available individual explanation.

In [ ]:
display(HTML(reading_room(room)))

## 10. Discoveries, false alarms and misses

In 2023, **M. Kerkez** and **M. Svilar** both started at 69 and later reached 82. The model ranked them fifth and thirtieth; FIFA potential ranked them fifty-second and 376th.

The same model ranked **Pablo Torre** first, but his observed peak in the window was 74. It also placed **A. Güler** sixty-second and missed him at top 50. These examples include discoveries, a false alarm and a miss.

Individual SHAP values connect these rankings to specific inputs. They sum, with the expected score, to the fitted raw score. The displayed bars are the six largest contributions; the score is a ranking index rather than a calibrated probability.

In [ ]:
display(HTML(case_comparison_html(study)))
plot_careers(study)
plot_case_shap(study)

## 11. How I selected the final model

I treated this as a ranking problem: given the same 50-player review budget, which approach finds more confirmed future 80+ players? I compared FIFA potential, compact learned controls, the player model, and extensions with current or forecast role profiles.

The 431-input player model had the strongest top-50 result in the four historical replays. I kept its CatBoost settings fixed for the feature-removal comparisons: 450 iterations, depth 5, learning rate 0.045, L2 regularisation 12, border count 64 and seed 42. Shallower trees and regularisation limit the complexity of each fitted interaction.

**I did not run an automated hyperparameter search or use these evaluation cohorts for early stopping.** This is an exploratory model/feature comparison on an archive studied previously; its replay periods are not an untouched final test set.

**I did not calibrate the output probabilities.** Sample weights emphasise the scouting use case, so I use the raw model output only to order players. A high ranking is not a stated percentage chance of reaching 80+. If I wanted probabilities, I would fit a calibration step on a separate mature validation window and check reliability and a proper probability score.

In [ ]:
display(HTML(selection_summary_html(study)))

## 12. What I found

**At a fixed review budget, adding player information improves the historical 80+ shortlist.** The gain is 22 versus 14 confirmed selections. Team and peer context produce the largest loss when removed.

The stricter pool of players never previously rated 70+ gives **23 versus 14** confirmed selections. For the rarer 85+ outcome, the model and potential each find **5**; there are only ten documented 85+ player-edition events in these cohorts.

I also tested forecast role-profile features. That extension finds 17 future 80+ selections at top 50, below the player model's 22. The improvement over potential comes from the player model; the role forecast did not improve this shortlist.

The scouting value is a better order for reviewing low-rated players. The next evaluation is a frozen pipeline on later-maturing cohorts, with match performance and playing time alongside game ratings.

In [ ]:
display(HTML(methods_html(study)))

## 13. The latest data-backed shortlist

I apply the selected player model to players still rated below 70, at any age, excluding previously observed 85+ players. The table shows the model's first 20 selections, their FIFA potential rank in the same eligible pool, and the inputs that most strongly support or oppose each model score.

FIFA potential is part of the ML model. Its value is interpreted together with the player's attributes, history, team and peers. The individual explanations separate the contribution of current potential and headroom from the contribution of all other inputs.

These are **unconfirmed prospective selections**. The data date and target editions are shown with the results. The bundled complete snapshot is dated **21 September 2025**; it is not a live November 2026 roster. The newer files checked below lack usable potential, so I do not replace it with invented or silently carried-forward values.

Each explanation uses native SHAP values from the selected model. Positive values raise the ranking score and negative values lower it. They describe the model's reasoning, rather than causes of future development.

In [ ]:
plot_current_priorities(study)
plot_current_player_reasons(study)
display(HTML(current_shortlist_html(study)))
display(HTML(newer_source_audit_html()))